# Phase 10 — the soft-prompt upper bound on decoherence

`Qwen/Qwen3-8B`, thinking off, no system message, query `what shall i do today` — phases 6–9's
configuration, so every reference band carries over.

**The question.** Nine phases rest on one claim: *prompt space cannot reach these behaviours*.
It is currently unsafe, because "prompt space cannot reach it" and "our search cannot find it"
are still the same sentence. The soft prompt separates them: every token sequence is
representable as a soft prompt, so soft-prompt performance **upper-bounds** anything GCG could
achieve at those slots.

This notebook is **stages 0–2** of `RECIPE.md`. Each is cheap and each can independently
invalidate what follows. Run in order; stop if one fails.

| stage | what it could invalidate |
|---|---|
| 0 | the pool-uniformity check reaches back to phase 3 |
| 1 | the objective's three demonstrated exits — a stronger optimiser finds all three |
| 2 | whether GCG has *ever* worked on this backbone (`NEXT-STEPS` item 2) |

Reference points, all on the raw entropy readout: clean baseline **0.683 ± 0.089** bits ·
hand-written battery ceiling **1.189** · L16 activation edit at s=1.0 **4.800** (the
activation-space ceiling, and the number the soft prompt is trying to approach from the input
side) · uniform vocabulary draw 12.645 · `log2(V)` 17.21.

In [1]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 129.2 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [2]:
# Environment — must run BEFORE anything imports huggingface_hub (phase 6 RECIPE stage 0).
# 1. HF_HUB_DISABLE_XET: without it the safetensors shards hang at 0 bytes.
# 2. HF_TOKEN from the Colab secrets vault, read early. Never print the token itself.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print("no colab secrets:", type(e).__name__)

HF_TOKEN present: True


In [3]:
# Load Qwen3-8B (bf16 where supported) — phase 6/7/8/9's exact backbone.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda:0")
model.eval(); model.requires_grad_(False)
dev    = model.device
LAYERS = model.model.layers
N_L    = model.config.num_hidden_layers
V      = model.config.vocab_size
EOS    = tokenizer.eos_token_id
print(f"{N_L} layers | d_model {model.config.hidden_size} | vocab {V} | "
      f"{sum(p.numel() for p in model.parameters())/1e9:.2f} B params | "
      f"{torch.cuda.memory_allocated()/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

36 layers | d_model 4096 | vocab 151936 | 8.19 B params | 15.3 GiB


## §0 — rig check, and the one line that could invalidate phases 3–8

Reproduce phase 9 §0 exactly: six queries, greedy 160, `H` = 0.703 / 0.527 / 0.743 / 0.730 /
0.639 / 0.648 and `distinct` = 0.588 / 0.670 / 0.638 / 0.575 / 0.663 / 0.644. Pool guard
151936 → **148023**. L16 CAA `‖V‖` 49.98, pairwise cosine 0.452, mean non-sink `‖h_L16‖` 89.77.

**Then the new part.** Phase 9 §9 found three rare glyphs each recurring in two independent
optimised triggers and computed P(≥3) ≈ 1e-5 *against the 148023-token pool*. Draw 10⁵ tokens
with the sampler the searches actually use and count distinct against the exact coupon-collector
expectation `n·(1 − (1 − 1/n)^m)`.

⚠ The RECIPE's "uniform gives ~85k distinct" is wrong: for n = 148023, m = 10⁵ the expectation is
**≈ 72.7k**. Use the computed value, not the quoted one.

In [6]:
# === §0a — rig check: phase 6 §2's entropy and distinctness, greedy 160 ===
import torch, torch.nn.functional as F, math, unicodedata, inspect, json, time
from collections import Counter

# only materialise the logits we score — a full [B, T, 151936] fp32 tensor is ~1 GB at B=8
_LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
        else "num_logits_to_keep")

QUERIES = [
    "what shall i do today",
    "recommend me a book",
    "how do I make friends in a new city?",
    "what should I get my brother for his birthday?",
    "tell me about bridges",
    "explain how suspension bridges work",
]
Q = QUERIES[0]                              # phase 6/7/8/9's query

def _chat(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)

def _ids(s):  return tokenizer(s, add_special_tokens=False).input_ids
def distinct_ratio(ids): return len(set(ids)) / max(1, len(ids))

@torch.no_grad()
def teacher_H(prompt_ids, ans_ids):
    # mean next-token entropy (bits) over the answer positions, teacher-forced
    seq = torch.tensor([list(prompt_ids) + list(ans_ids)], device=dev)
    lg  = model(seq, **{_LTK: len(ans_ids) + 1}).logits[0, :-1].float()
    P   = lg.softmax(-1)
    H   = -(P * P.clamp_min(1e-12).log2()).sum(-1).mean().item()
    del lg, P
    return H

PH6_H    = [0.703, 0.527, 0.743, 0.730, 0.639, 0.648]
PH6_DIST = [0.588, 0.670, 0.638, 0.575, 0.663, 0.644]

RIG = {}
w = max(len(q) for q in QUERIES)
print(f"{'query':<{w}}  {'H':>6} {'(ph9)':>7}  {'dist':>6} {'(ph9)':>7}   T")
for q, h6, d6 in zip(QUERIES, PH6_H, PH6_DIST):
    p = _ids(_chat(q))
    with torch.no_grad():
        g = model.generate(torch.tensor([p], device=dev), max_new_tokens=160,
                           do_sample=False, pad_token_id=EOS)[0]
    a = g[len(p):].tolist()
    H, d = teacher_H(p, a), distinct_ratio(a)
    RIG[q] = dict(H=H, distinct=d, T=len(a),
                  answer=tokenizer.decode(a, skip_special_tokens=True))
    print(f"{q:<{w}}  {H:>6.3f} {h6:>7.3f}  {d:>6.3f} {d6:>7.3f}  {len(a):>4}")
    torch.cuda.empty_cache()

NORMAL_H    = (min(v['H'] for v in RIG.values()),        max(v['H'] for v in RIG.values()))
NORMAL_DIST = (min(v['distinct'] for v in RIG.values()), max(v['distinct'] for v in RIG.values()))
print(f"\nnormal band: H {NORMAL_H[0]:.3f}-{NORMAL_H[1]:.3f} bits | "
      f"distinct {NORMAL_DIST[0]:.3f}-{NORMAL_DIST[1]:.3f}")
print(f"entropy ceiling log2(V) = {math.log2(V):.2f} bits")

_dH = max(abs(RIG[q]['H'] - h) for q, h in zip(QUERIES, PH6_H))
_dD = max(abs(RIG[q]['distinct'] - d) for q, d in zip(QUERIES, PH6_DIST))
print(f"max |ΔH| vs phase 9: {_dH:.4f} | max |Δdistinct|: {_dD:.4f}"
      f"   {'OK' if _dH < 0.02 and _dD < 0.02 else '*** RIG MISMATCH ***'}")

query                                                H   (ph9)    dist   (ph9)   T
what shall i do today                            0.703   0.703   0.588   0.588   160
recommend me a book                              0.527   0.527   0.670   0.670    94
how do I make friends in a new city?             0.743   0.743   0.637   0.638   160
what should I get my brother for his birthday?   0.730   0.730   0.575   0.575   160
tell me about bridges                            0.639   0.639   0.662   0.663   160
explain how suspension bridges work              0.648   0.648   0.644   0.644   160

normal band: H 0.527-0.743 bits | distinct 0.575-0.670
entropy ceiling log2(V) = 17.21 bits
max |ΔH| vs phase 9: 0.0004 | max |Δdistinct|: 0.0005   OK


In [7]:
# === §0b — the pools, exactly as every phase from 3 onward builds them ===
import torch, unicodedata

TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc","Cs","Co") for c in s):
        usable[i] = False

_E   = model.model.embed_tokens.weight.float()
WEAK = (_E - _E.mean(0, keepdim=True)).norm(dim=-1).cpu()
del _E; torch.cuda.empty_cache()

USABLE  = torch.nonzero(usable).squeeze(-1)         # the "148023-token pool"
WEAK4K  = USABLE[WEAK[USABLE].argsort()[:4096]]     # phase 2's weakest-norm pool
WEAKEST = int(WEAK4K[0])
print(f"vocab {V} -> usable {len(USABLE)}  (phase 9: 148023)"
      f"   {'OK' if len(USABLE) == 148023 else '*** POOL GUARD MISMATCH ***'}")
print(f"WEAK4K {len(WEAK4K)} | weakest token id {WEAKEST} {TOKSTR[WEAKEST]!r} "
      f"(norm {WEAK[WEAKEST]:.3f})")

vocab 151936 -> usable 148023  (phase 9: 148023)   OK
WEAK4K 4096 | weakest token id 143335 'ספטמ' (norm 0.157)


In [8]:
# === §0c — ⁂ is the pool uniform?  (NEXT-STEPS item 6 / RECIPE stage 0) ===
# The sampler every search actually uses is pool[torch.randint(0, len(pool), (k,), gen)].
# Drive it directly for m = 1e5 draws and compare against the exact coupon-collector mean.
import torch, math
from collections import Counter

def pool_draw(pool, m, seed=0):
    g = torch.Generator().manual_seed(seed)
    return pool[torch.randint(0, len(pool), (m,), generator=g)]

M = 100_000
POOLS = {"USABLE (148023)": USABLE, "WEAK4K (4096)": WEAK4K}
UNIF = {}
print(f"{'pool':<18} {'distinct':>9} {'expected':>9} {'ratio':>7} {'chi2/df':>9} {'top-1 count':>12}")
for name, pool in POOLS.items():
    n = len(pool)
    draws = pool_draw(pool, M, seed=0).tolist()
    c = Counter(draws)
    exp_distinct = n * (1 - (1 - 1/n)**M)
    # Pearson chi-square against uniform over all n cells, empty cells included
    e = M / n
    chi2 = (sum((v - e)**2 for v in c.values()) + (n - len(c)) * e**2) / e
    df = n - 1
    top = c.most_common(3)
    UNIF[name] = dict(n=n, m=M, distinct=len(c), expected=exp_distinct,
                      chi2_over_df=chi2/df, top=[(int(t), int(v), TOKSTR[t]) for t, v in top])
    print(f"{name:<18} {len(c):>9} {exp_distinct:>9.0f} {len(c)/exp_distinct:>7.3f} "
          f"{chi2/df:>9.3f} {top[0][1]:>6}  {TOKSTR[top[0][0]]!r}")

print("\nchi2/df ≈ 1.0 and ratio ≈ 1.000 ⇒ uniform. A flat rank-frequency curve follows.")
print("rank-frequency, USABLE, ranks 1/10/100/1000/10000:")
_c = sorted(Counter(pool_draw(USABLE, M, seed=0).tolist()).values(), reverse=True)
for r in (1, 10, 100, 1000, 10000):
    if r <= len(_c): print(f"   rank {r:>6}: {_c[r-1]}")

pool                distinct  expected   ratio   chi2/df  top-1 count
USABLE (148023)        72701     72699   1.000     0.998      6  ')row'
WEAK4K (4096)           4096      4096   1.000     0.998     43  '쭐'

chi2/df ≈ 1.0 and ratio ≈ 1.000 ⇒ uniform. A flat rank-frequency curve follows.
rank-frequency, USABLE, ranks 1/10/100/1000/10000:
   rank      1: 6
   rank     10: 6
   rank    100: 5
   rank   1000: 3
   rank  10000: 2


In [9]:
# === §0d — ⁂ where phase 9 §9's collisions actually came from ===
# §4's length sweep called hillclimb(...) with pool=None, and hillclimb's default is
#     pool = WEAK4K if pool is None else pool
# so all fourteen length-sweep triggers were drawn from 4096 tokens, not 148023.
# Phase 9 §9 computed its P(≥3) ≈ 1e-5 against the 148023 pool. Check the arithmetic.
import json, math
from collections import Counter

try:
    _ph9 = json.load(open("phase9_decoherence.json"))
except FileNotFoundError:
    _ph9 = None
    print("phase9_decoherence.json not uploaded — skipping (upload it to rerun this cell)")

POOLCHK = None
if _ph9:
    slots = [(k, t) for k, v in _ph9["length_sweep"].items() for t in v["trigger"]]
    ids   = [t for _, t in slots]
    c     = Counter(ids)
    coll  = sum(v - 1 for v in c.values() if v > 1)          # excess occupancy
    pairs = len(ids) * (len(ids) - 1) // 2
    w4    = set(WEAK4K.tolist())
    in_w4 = sum(t in w4 for t in ids)
    print(f"length-sweep slots {len(ids)} | distinct {len(c)} | collisions {coll}")
    print(f"slots drawn from WEAK4K: {in_w4}/{len(ids)}"
          f"   {'CONFIRMED' if in_w4 == len(ids) else '*** MIXED POOLS ***'}")
    print()
    print(f"{'pool assumed':<20} {'E[collisions]':>14} {'P(>=obs)':>10}")
    for nm, n in (("WEAK4K (4096)", 4096), ("USABLE (148023)", 148023)):
        lam = pairs / n
        # Poisson tail P(X >= coll)
        p = 1 - sum(math.exp(-lam) * lam**i / math.factorial(i) for i in range(coll))
        print(f"{nm:<20} {lam:>14.3f} {p:>10.3g}")
    POOLCHK = dict(slots=len(ids), distinct=len(c), collisions=coll, pairs=pairs,
                   in_weak4k=in_w4,
                   dupes=[dict(id=int(t), n=int(v), tok=TOKSTR[t],
                               where=sorted({k for k, tt in slots if tt == t}))
                          for t, v in c.items() if v > 1])
    print("\n⁂ If in_weak4k == slots and E[collisions|WEAK4K] ≈ observed, phase 9 §9's alarm is a")
    print("  pool-size bookkeeping error, not evidence of non-uniformity, and phases 3–8 stand.")

length-sweep slots 254 | distinct 243 | collisions 11
slots drawn from WEAK4K: 254/254   CONFIRMED

pool assumed          E[collisions]   P(>=obs)
WEAK4K (4096)                 7.844      0.169
USABLE (148023)               0.217   1.11e-15

⁂ If in_weak4k == slots and E[collisions|WEAK4K] ≈ observed, phase 9 §9's alarm is a
  pool-size bookkeeping error, not evidence of non-uniformity, and phases 3–8 stand.


In [10]:
# === §0e — the L16 CAA vector and the activation-space ceiling ===
import torch, torch.nn.functional as F

L_CAA = 16
CTX   = "The word is"
NEGS  = [" cat", " chair", " cloud", " music", " running", " table", " coffee", " window"]

class _Stop(Exception): pass
def _capture(store):
    def hook(mod, args, kwargs):
        store.append(args[0] if args else kwargs["hidden_states"]); raise _Stop
    return hook

@torch.no_grad()
def h_last(text, L):
    store = []
    hd = LAYERS[L].register_forward_pre_hook(_capture(store), with_kwargs=True)
    try:    model(torch.tensor([_ids(text)], device=dev), use_cache=False)
    except _Stop: pass
    finally: hd.remove()
    return store[0][0, -1].float()

vs = [h_last(f"{CTX} bridge", L_CAA) - h_last(f"{CTX}{n}", L_CAA) for n in NEGS]
V_CAA = torch.stack(vs).mean(0)
_n = F.normalize(torch.stack(vs), dim=-1)
pair = ((_n @ _n.T).sum() - len(vs)) / (len(vs)*(len(vs)-1))

@torch.no_grad()
def mean_nonsink(L, prompt_ids):
    store = []
    hd = LAYERS[L].register_forward_pre_hook(_capture(store), with_kwargs=True)
    try:    model(torch.tensor([prompt_ids], device=dev), use_cache=False)
    except _Stop: pass
    finally: hd.remove()
    return store[0][0, 1:].float().norm(dim=-1).mean().item()

CLEAN_Q = _ids(_chat(Q))
MNS = mean_nonsink(L_CAA, CLEAN_Q)
print(f"||V_CAA|| = {V_CAA.norm():.2f} (ph9 49.98) | pairwise cos {pair:.3f} (ph9 0.452) | "
      f"mean non-sink ||h_L16|| = {MNS:.2f} (ph9 89.77)")

def steer_hook(v, alpha):
    def hook(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        add = (alpha * v).to(h.dtype)
        h = h.clone()
        if h.shape[1] > 1: h[:, 1:] += add          # prefill: skip the attention sink
        else:              h += add                 # decode: never position 0
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h; return args, kwargs
    return hook

||V_CAA|| = 49.98 (ph9 49.98) | pairwise cos 0.452 (ph9 0.452) | mean non-sink ||h_L16|| = 89.77 (ph9 89.77)


## §1 — the readout, then the three repairs

Phase 9's readout, ported unchanged so every stored number stays comparable, followed by
stage 1's repairs. The repairs are measured *against* the phase 9 numbers, not substituted for
them silently.

| blind spot | phase 9 evidence | repair |
|---|---|---|
| **language** | Arabic/Hebrew score 1.7–2.6 against English 0.7–1.2; the surviving trigger's whole effect is a language hijack | `H_excess = H − H_baseline(detected script)` |
| **enumeration** | ten near-synonyms score high with `distinct` 0.714 — neither gate fires | type-token ratio over **embedding clusters**, not tokens |
| **type C** | outputs that fabricate the user's message score *near baseline* because they are fluent | quoted-span premise detector, reported as a **flag beside** the objective |

In [11]:
# === §1 — phase 9's readout, ported unchanged ===
import torch, unicodedata, math
from collections import Counter

D_GATE, REP_GATE = 0.75, 3

def _script(ch):
    try:    return unicodedata.name(ch).split(" ")[0]
    except ValueError: return "?"

def surface(text):
    letters = [c for c in text if c.isalpha()]
    sc = [_script(c) for c in letters]
    if not sc: return 0.0, 0.0
    nonlatin = sum(s != "LATIN" for s in sc) / len(sc)
    switch   = (sum(a != b for a, b in zip(sc, sc[1:])) / max(1, len(sc) - 1))
    return nonlatin, switch

def dominant_script(text):
    # the repair's language proxy: modal script over alphabetic characters
    letters = [c for c in text if c.isalpha()]
    if not letters: return "NONE"
    sc = Counter(_script(c) for c in letters)
    s, n = sc.most_common(1)[0]
    return s if n / len(letters) >= 0.5 else "MIXED"

def max_ngram(ids, n=4):
    if len(ids) < n: return 0
    return Counter(tuple(ids[i:i+n]) for i in range(len(ids)-n+1)).most_common(1)[0][1]

@torch.no_grad()
def nll_under(prompt_ids, ans_ids):
    seq = torch.tensor([list(prompt_ids) + list(ans_ids)], device=dev)
    lp  = model(seq, **{_LTK: len(ans_ids) + 1}).logits[0, :-1].float().log_softmax(-1)
    t   = torch.tensor(ans_ids, device=dev)
    out = -lp.gather(-1, t[:, None]).mean().item()
    del lp
    return out

@torch.no_grad()
def gen(prompt_ids, n_new=96, seed=0, greedy=False, temp=1.0):
    ids = torch.tensor([list(prompt_ids)], device=dev)
    if greedy:
        g = model.generate(ids, max_new_tokens=n_new, do_sample=False, pad_token_id=EOS)[0]
    else:
        torch.manual_seed(seed)
        g = model.generate(ids, max_new_tokens=n_new, do_sample=True, temperature=temp,
                           top_p=1.0, top_k=0, pad_token_id=EOS)[0]
    return g[ids.shape[1]:].tolist()

def slop_of(H, distinct, rep4):
    return H - 8.0*max(0.0, D_GATE - distinct) - 0.5*max(0, rep4 - REP_GATE)

CLEAN = {Q: CLEAN_Q}

def measure(prompt_ids, ans_ids, q=None):
    txt = tokenizer.decode(ans_ids, skip_special_tokens=True)
    H   = teacher_H(prompt_ids, ans_ids)
    d   = distinct_ratio(ans_ids); r4 = max_ngram(ans_ids)
    nl, sw = surface(txt)
    clean = CLEAN.setdefault(q or Q, _ids(_chat(q or Q)))
    return dict(H=H, distinct=d, rep4=r4, nll_clean=nll_under(clean, ans_ids),
                nonlatin=nl, switch=sw, T=len(ans_ids), script=dominant_script(txt),
                slop=slop_of(H, d, r4), text=txt)

def mean_rows(rows):
    keys = ("H","distinct","rep4","nll_clean","nonlatin","switch","T","slop")
    return {k: sum(r[k] for r in rows)/len(rows) for k in keys}

HDR = (f"{'tag':<34} {'H':>6} {'dist':>5} {'rep4':>4} {'nllC':>6} "
       f"{'nonL':>5} {'swch':>5} {'slop':>7}")
def show(tag, m, text=None, width=110):
    print(f"{tag:<34} {m['H']:>6.3f} {m['distinct']:>5.2f} {m['rep4']:>4.0f} "
          f"{m['nll_clean']:>6.2f} {m['nonlatin']:>5.2f} {m['switch']:>5.2f} {m['slop']:>7.3f}")
    if text is not None:
        print(f"    {text[:width]!r}")

def scaffold(q, position):
    s = _chat(q); i = s.index(q)
    head, tail = (s[:i], s[i:]) if position == "prefix" else (s[:i+len(q)], s[i+len(q):])
    return _ids(head), _ids(tail)

PRE_S, SUF_S = scaffold(Q, "suffix")
PRE_P, SUF_P = scaffold(Q, "prefix")
print(f"suffix scaffold {len(PRE_S)} + {len(SUF_S)} | prefix {len(PRE_P)} + {len(SUF_P)}"
      f" | clean prompt {len(CLEAN[Q])} tokens   (ph9: 8+9, 3+14, 17)")

# baseline, 10 samples, fixed seeds — phase 9 measured 0.683 ± 0.089
_rows = [measure(CLEAN[Q], gen(CLEAN[Q], 96, i), q=Q) for i in range(10)]
_mu   = sum(r["H"] for r in _rows)/10
_sd   = (sum((r["H"]-_mu)**2 for r in _rows)/9) ** .5
BASE_EN = dict(mu=_mu, sd=_sd, rows=[{k: r[k] for k in
              ("H","distinct","rep4","nll_clean","nonlatin","switch","script","text")}
              for r in _rows])
print(f"\nEnglish baseline H = {_mu:.3f} ± {_sd:.3f}  (ph9: 0.683 ± 0.089)")
print(HDR); show("[no trigger, sampled T=1.0]", mean_rows(_rows), _rows[0]["text"])

suffix scaffold 8 + 9 | prefix 3 + 14 | clean prompt 17 tokens   (ph9: 8+9, 3+14, 17)

English baseline H = 0.674 ± 0.088  (ph9: 0.683 ± 0.089)
tag                                     H  dist rep4   nllC  nonL  swch    slop
[no trigger, sampled T=1.0]         0.674  0.71    1   0.45  0.00  0.00   0.332
    "That's a great question! 🌞 What would you like to do today? Here are a few fun and meaningful ideas to help yo"


### §1a — the language control

Phase 9 owes this and never ran it. Score a **fluent, on-topic, forced-language** answer to the
clean query with no trigger. Two forcings per language, because a bare prefill does not guarantee
fluency or on-topicness:

- **prefill** — append the opening words to the assistant turn (`أعتقد`, `אני חושב`, `我觉得`)
- **instructed** — append `Answer in Arabic.` to the user turn, then prefill

Entropy is measured under the **clean** prompt in both cases, so the number is a property of the
answer, comparable with the baseline. If forced-Arabic lands near **1.7**, phase 9 §8's one
surviving trigger has no effect beyond language switching and should be reported as zero.

In [12]:
# === §1a — the language control ===
import torch

LANGS = [
    ("Arabic",  "Answer in Arabic.",  "أعتقد"),
    ("Hebrew",  "Answer in Hebrew.",  "אני חושב"),
    ("Chinese", "Answer in Chinese.", "我觉得"),
]

def forced(instruct, prefill, seed=0, n_new=96):
    # generate with a prefilled assistant turn; score the whole answer under the CLEAN prompt,
    # so the number is a property of the answer and stays comparable with the baseline
    uq   = Q if instruct is None else f"{Q}\n\n{instruct}"
    pid  = _ids(_chat(uq) + prefill)
    full = _ids(prefill) + gen(pid, n_new, seed=seed)
    return measure(CLEAN[Q], full, q=Q)

LANGCTL = {}
print(HDR)
show("English/baseline  [LATIN]", mean_rows(_rows), _rows[0]["text"])
LANGCTL["English/baseline"] = dict(mean=mean_rows(_rows), script="LATIN",
                                   texts=[r["text"] for r in _rows[:3]])
for name, instruct, prefill in LANGS:
    for mode, ins in (("prefill", None), ("instructed", instruct)):
        rows = [forced(ins, prefill, seed=i) for i in range(3)]
        m, tag = mean_rows(rows), f"{name}/{mode}"
        LANGCTL[tag] = dict(mean=m, script=rows[0]["script"],
                            texts=[r["text"] for r in rows])
        show(f"{tag}  [{rows[0]['script'][:8]}]", m, rows[0]["text"])

# language-matched baselines, keyed by dominant script
H_BASE = {}
for tag, r in LANGCTL.items():
    H_BASE.setdefault(r["script"], []).append(r["mean"]["H"])
H_BASE = {k: sum(v)/len(v) for k, v in H_BASE.items()}
H_BASE.setdefault("LATIN", BASE_EN["mu"])
print("\nlanguage-matched baselines (H by dominant script):")
for k, v in sorted(H_BASE.items(), key=lambda kv: -kv[1]):
    print(f"   {k:<10} {v:.3f}")

def H_excess(m):
    return m["H"] - H_BASE.get(m.get("script", "LATIN"), BASE_EN["mu"])

print(f"\n⁂ phase 9 §8's surviving trigger scored 1.721 raw. Against the Arabic baseline above,")
print(f"  its excess is what matters — if that is ~0, the effect is language switching alone.")

tag                                     H  dist rep4   nllC  nonL  swch    slop
English/baseline  [LATIN]           0.674  0.71    1   0.45  0.00  0.00   0.332
    "That's a great question! 🌞 What would you like to do today? Here are a few fun and meaningful ideas to help yo"
Arabic/prefill  [ARABIC]            1.904  0.80    1   1.71  0.84  0.01   1.863
    'أعتقد أنك بحاجة إلى مزيد من التفاصيل لمساعدتك في تحديد ما يجب أن تفعله اليوم. هل есть هدف أو أهداف معينة ترغبي'
Arabic/instructed  [ARABIC]         1.059  0.78    1   1.17  1.00  0.00   0.841
    'أعتقد أنك بحاجة إلى تحديد ما تريده من اليوم. هل ترغب في أن أقترح لك بعض الأنشطة أو الأفكار؟'
Hebrew/prefill  [LATIN]             3.280  0.83    1   2.63  0.67  0.03   3.280
    'אני חושב על כמה פעילויות מעניינות שייתנו לך DAY开心:\n\n**1. חוויה שלא נסעת אליכם (Yet)**\n- across the world, try '
Hebrew/instructed  [HEBREW]         3.254  0.84    1   3.03  0.97  0.02   3.254
    'אני חושב על מה שיקרה היום. מה תרצה לעשות?'
Chinese/prefill  [CJ

In [13]:
# === §1a2 — rebuild the baselines from FLUENT MONOLINGUAL forcings only ===
# Hebrew/prefill is code-mixed ('DAY开心', nonlatin 0.67) and dominant_script called it LATIN,
# which poisoned the LATIN baseline at 1.977. A code-mixed answer is genuine decoherence, not a
# free exit, so it must not enter the baseline. Keep only forcings that stayed in one script.
# Also print T: a short answer's mean entropy is dominated by its opening tokens, and phase 6
# §10's answer-length warning applies to exactly this comparison.
print(f"{'tag':<22} {'script':<8} {'nonL':>5} {'T':>5} {'H':>7}   verdict")
MONO = {}
for tag, r in LANGCTL.items():
    m = r["mean"]; nl = m["nonlatin"]
    mono = (nl >= 0.90) or (nl <= 0.10)
    print(f"{tag:<22} {r['script']:<8} {nl:>5.2f} {m['T']:>5.0f} {m['H']:>7.3f}   "
          f"{'monolingual' if mono else '*** code-mixed — excluded ***'}")
    if mono: MONO.setdefault(r["script"], []).append(m["H"])

H_BASE = {k: sum(v)/len(v) for k, v in MONO.items()}
H_BASE["LATIN"] = BASE_EN["mu"]                      # English, 10 samples, nonlatin 0.00
print("\nlanguage-matched baselines (fluent monolingual only):")
for k, v in sorted(H_BASE.items(), key=lambda kv: -kv[1]):
    print(f"   {k:<8} {v:.3f}")

_free = max(H_BASE.values())
print(f"\n⁂ the cheapest free exit is {max(H_BASE, key=H_BASE.get)} at {_free:.3f} bits, against")
print(f"  English {H_BASE['LATIN']:.3f}, the hand-written battery ceiling 1.189, phase 9's one")
print(f"  surviving trigger 1.721, and the L16 activation ceiling 4.800.")
print(f"  {(_free - H_BASE['LATIN']) / (4.800 - H_BASE['LATIN']):.0%} of the way from baseline to"
      f" the activation-space ceiling is buyable with no trigger at all.")
print("\n⚠ length caveat: compare T across rows before trusting any of this. A 15-token answer")
print("  and a 96-token answer are not on the same scale (phase 6 §10).")


tag                    script    nonL     T       H   verdict
English/baseline       LATIN     0.00    96   0.674   monolingual
Arabic/prefill         ARABIC    0.84    98   1.904   *** code-mixed — excluded ***
Arabic/instructed      ARABIC    1.00    77   1.059   monolingual
Hebrew/prefill         LATIN     0.67    98   3.280   *** code-mixed — excluded ***
Hebrew/instructed      HEBREW    0.97    54   3.254   monolingual
Chinese/prefill        CJK       1.00    90   1.216   monolingual
Chinese/instructed     CJK       1.00    87   1.116   monolingual

language-matched baselines (fluent monolingual only):
   HEBREW   3.254
   CJK      1.166
   ARABIC   1.059
   LATIN    0.674

⁂ the cheapest free exit is HEBREW at 3.254 bits, against
  English 0.674, the hand-written battery ceiling 1.189, phase 9's one
  surviving trigger 1.721, and the L16 activation ceiling 4.800.
  63% of the way from baseline to the activation-space ceiling is buyable with no trigger at all.

⚠ length caveat: co

### §1b–d — the enumeration gate, the premise detector, the mode coding

- **1b.** Cluster the answer's unique tokens by embedding cosine and take the type-token ratio
  over *clusters*. Calibrated on phase 9's stored texts: the k=53 enumeration answer must fail
  the new gate and §0's six normal answers must pass. The threshold is swept, not guessed.
- **1c.** Ten lines: every quoted span in the answer, tested for presence in the user turn.
  Absent ⇒ `confabulated` (type C); present ⇒ `reading` mode. Reported as a **flag**, never folded
  into the objective.
- **1d.** Label every stored sample **normal / L / S / C** and report the distribution beside every
  mean. The k=4 search winner's 1.097 hides one genuine type-S sample among nine near-baseline.

In [14]:
# === §1b — the enumeration gate: type-token ratio over embedding clusters ===
import torch, torch.nn.functional as F

_EMB = model.model.embed_tokens.weight

@torch.no_grad()
def cluster_ttr(ids, tau=0.60):
    # greedy single-link agglomeration of the answer's UNIQUE tokens by embedding cosine
    uniq = sorted(set(ids))
    if not uniq: return 0.0, 0
    e = F.normalize(_EMB[torch.tensor(uniq, device=dev)].float(), dim=-1)
    S = (e @ e.T) >= tau
    seen, comp = set(), 0
    for i in range(len(uniq)):
        if i in seen: continue
        stack, comp = [i], comp + 1
        while stack:
            j = stack.pop()
            if j in seen: continue
            seen.add(j)
            stack.extend(torch.nonzero(S[j]).squeeze(-1).tolist())
    return comp / len(ids), comp

# calibration set: phase 9's six normal answers (must PASS) vs its enumeration answer (must FAIL)
CAL_TEXTS = {}
if _ph9:
    for q, v in _ph9["rig"].items():
        CAL_TEXTS[f"normal/{q[:28]}"] = (v["answer"], "pass")
    _enum = _ph9["search"]["phase6 k=53 suffix"]["best"]["text"]
    CAL_TEXTS["ENUMERATION/k=53"] = (_enum, "fail")
    for tag in ("suffix|k=1", "prefix|k=2"):
        CAL_TEXTS[f"sweep/{tag}"] = (_ph9["length_sweep"][tag]["best"]["text"], "?")

print(f"{'tau':>5}  {'sep':>7}  {'min pass':>9}  {'max fail':>9}")
BEST_TAU, GATE_C = None, None
for tau in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80):
    p, f_ = [], []
    for tag, (txt, want) in CAL_TEXTS.items():
        r, _ = cluster_ttr(_ids(txt), tau)
        (p if want == "pass" else f_ if want == "fail" else []).append(r)
    if not p or not f_: continue
    sep = min(p) - max(f_)
    print(f"{tau:>5.2f}  {sep:>7.3f}  {min(p):>9.3f}  {max(f_):>9.3f}")
    if BEST_TAU is None or sep > BEST_TAU[1]: BEST_TAU = (tau, sep)

if BEST_TAU:
    TAU = BEST_TAU[0]
    ps = [cluster_ttr(_ids(t), TAU)[0] for t, w in CAL_TEXTS.values() if w == "pass"]
    fs = [cluster_ttr(_ids(t), TAU)[0] for t, w in CAL_TEXTS.values() if w == "fail"]
    GATE_C = (min(ps) + max(fs)) / 2
    print(f"\nchosen tau {TAU} | cluster-TTR gate {GATE_C:.3f} | separation {BEST_TAU[1]:+.3f}")
    if BEST_TAU[1] <= 0:
        print("*** the cluster gate does NOT separate enumeration from normal — report as a")
        print("    negative and do not fold it into the objective (RECIPE stage 1b unmet) ***")
else:
    TAU, GATE_C = 0.60, 0.0
    print("no calibration data (phase9 json missing)")

  tau      sep   min pass   max fail
 0.30   -0.198      0.469      0.667
 0.40   -0.162      0.525      0.688
 0.50   -0.190      0.550      0.740
 0.60   -0.171      0.569      0.740
 0.70   -0.165      0.575      0.740
 0.80   -0.165      0.575      0.740

chosen tau 0.4 | cluster-TTR gate 0.606 | separation -0.162
*** the cluster gate does NOT separate enumeration from normal — report as a
    negative and do not fold it into the objective (RECIPE stage 1b unmet) ***


In [15]:
# === §1b2 — enumeration is LOCAL, so measure it locally ===
# Global cluster-TTR failed at every tau (separation -0.16 to -0.20) and failed in the WRONG
# DIRECTION: the enumeration answer scored HIGHER than the normal ones. Two reasons, both
# fatal to the RECIPE's version of the repair:
#   1. TTR of any flavour measures vocabulary richness, not repetition of meaning. Normal English
#      prose repeats function words heavily and so scores LOW; a run of rare Chinese nouns scores
#      HIGH. The gate is anti-correlated with what it was meant to detect.
#   2. The calibration was not language-matched — enumeration answer Chinese, controls English.
# A synonym run is a LOCAL property: adjacent items are semantically close. Measure that instead,
# and calibrate against a language-matched normal answer.
import torch, torch.nn.functional as F

@torch.no_grad()
def adjacent_sim(ids, win=10):
    # mean cosine between adjacent token embeddings, and the max over sliding windows
    ids = [t for t in ids if TOKSTR[t].strip()]
    if len(ids) < win + 1: return 0.0, 0.0
    e = F.normalize(_EMB[torch.tensor(ids, device=dev)].float(), dim=-1)
    adj = (e[:-1] * e[1:]).sum(-1)                     # [T-1]
    win_mu = adj.unfold(0, win, 1).mean(-1)            # [T-win]
    return adj.mean().item(), win_mu.max().item()

CAL2 = {}
if _ph9:
    CAL2["ENUM/k=53 (zh)"]  = (_ph9["search"]["phase6 k=53 suffix"]["best"]["text"], "fail")
CAL2["normal/zh instructed"] = (LANGCTL["Chinese/instructed"]["texts"][0], "pass")
CAL2["normal/zh prefill"]    = (LANGCTL["Chinese/prefill"]["texts"][0],    "pass")
CAL2["normal/ar instructed"] = (LANGCTL["Arabic/instructed"]["texts"][0],  "pass")
CAL2["normal/he instructed"] = (LANGCTL["Hebrew/instructed"]["texts"][0],  "pass")
for i, (q, v) in enumerate(list(RIG.items())[:3]):
    CAL2[f"normal/en {i}"] = (v["answer"], "pass")

print(f"{'text':<26} {'want':>5} {'adj_mu':>8} {'win_max':>8}")
rows = {}
for tag, (txt, want) in CAL2.items():
    a, w = adjacent_sim(_ids(txt))
    rows[tag] = (want, a, w)
    print(f"{tag:<26} {want:>5} {a:>8.3f} {w:>8.3f}")

for name, idx in (("adj_mu", 1), ("win_max", 2)):
    p = [v[idx] for v in rows.values() if v[0] == "pass"]
    f_ = [v[idx] for v in rows.values() if v[0] == "fail"]
    if p and f_:
        sep = min(f_) - max(p)                       # enumeration should score HIGHER
        print(f"\n{name}: enumeration {min(f_):.3f} vs normal max {max(p):.3f} "
              f"→ separation {sep:+.3f}  {'WORKS' if sep > 0 else 'FAILS'}")

# language-matched comparison is the point: the zh normal answer is itself a numbered list, so
# 'enumeration' and 'a helpful list' may not be separable at all on this query.
print("\nnote: the model's normal answer to this query IS a list, in every language tested.")


text                        want   adj_mu  win_max
ENUM/k=53 (zh)              fail    0.047    0.072
normal/zh instructed        pass    0.049    0.067
normal/zh prefill           pass    0.053    0.064
normal/ar instructed        pass    0.044    0.060
normal/he instructed        pass    0.030    0.035
normal/en 0                 pass    0.072    0.111
normal/en 1                 pass    0.067    0.109
normal/en 2                 pass    0.061    0.117

adj_mu: enumeration 0.047 vs normal max 0.072 → separation -0.025  FAILS

win_max: enumeration 0.072 vs normal max 0.117 → separation -0.045  FAILS

note: the model's normal answer to this query IS a list, in every language tested.


In [16]:
# === §1c — the premise-fidelity detector (type C), and §1d the mode coding ===
import re, unicodedata

_QUOTE = re.compile(r'"([^"]{4,120})"|“([^”]{4,120})”|«([^»]{4,120})»'
                    r'|「([^」]{4,120})」|『([^』]{4,120})』|\'([^\']{4,120})\'')

def _norm(s):
    s = unicodedata.normalize("NFKC", s).lower()
    return "".join(c for c in s if not unicodedata.category(c).startswith(("P", "Z", "C")))

def premise_flag(answer_text, prompt_text):
    p = _norm(prompt_text)
    spans = [next(g for g in m.groups() if g) for m in _QUOTE.finditer(answer_text)]
    spans = [s for s in spans if len(_norm(s)) >= 4]
    if not spans: return "none", []
    present = [s for s in spans if _norm(s) in p]
    absent  = [s for s in spans if _norm(s) not in p]
    return ("confabulated" if absent else "reading"), (absent or present)

def mode_of(m, prompt_text, base_mu=None, base_sd=None):
    # normal / L (loop) / S (slop) / C (confabulated premise)
    base_mu = BASE_EN["mu"] if base_mu is None else base_mu
    base_sd = BASE_EN["sd"] if base_sd is None else base_sd
    flag, _ = premise_flag(m["text"], prompt_text)
    if flag == "confabulated": return "C"
    if m["distinct"] < D_GATE or m["rep4"] > REP_GATE: return "L"
    ex = m["H"] - H_BASE.get(m.get("script", "LATIN"), base_mu)
    if ex > 2 * base_sd: return "S"
    return "normal"

# self-test on the two type-C cases phase 9 named, plus a reading-mode case
_PROMPT_TXT = _chat(Q)
_tests = [
    ('The model says "Tumblr doesn\'t even have product R&D — what should I do today?" here.',
     "confabulated"),
    ('You asked "what shall i do today", so here are some ideas.', "reading"),
    ('No quotes at all in this answer.', "none"),
]
print("premise detector self-test:")
for t, want in _tests:
    got, spans = premise_flag(t, _PROMPT_TXT)
    print(f"   {'OK ' if got == want else '*** FAIL ***'} {got:<13} want {want:<13} {t[:56]!r}")

# §1d — code the modes over everything phase 9 stored
MODES = {}
if _ph9:
    import collections
    def _code_block(name, items):
        cnt = collections.Counter()
        for tag, texts in items:
            for txt in texts:
                ids = _ids(txt)
                m = dict(H=teacher_H(CLEAN[Q], ids), distinct=distinct_ratio(ids),
                         rep4=max_ngram(ids), text=txt, script=dominant_script(txt))
                cnt[mode_of(m, _PROMPT_TXT)] += 1
        MODES[name] = dict(cnt)
        tot = sum(cnt.values())
        print(f"   {name:<24} n={tot:<4} " +
              "  ".join(f"{k} {v} ({v/tot:.0%})" for k, v in cnt.most_common()))

    print("\nmode distribution over phase 9's stored outputs:")
    _code_block("battery (42 cells x3)",
                [(k, v["samples"]) for k, v in _ph9["battery"].items()])
    _code_block("length sweep (best)",
                [(k, [v["best"]["text"]]) for k, v in _ph9["length_sweep"].items()])
    _code_block("search arms (best)",
                [(k, [v["best"]["text"]]) for k, v in _ph9["search"].items()])
    try:
        _surv = json.load(open("phase9_surviving_trigger_10x.json"))
        _txts = _surv.get("samples") or _surv.get("texts") or []
        if _txts: _code_block("surviving trigger (10x)", [("surv", _txts)])
    except Exception as e:
        print("   (phase9_surviving_trigger_10x.json not loaded:", type(e).__name__, ")")

premise detector self-test:
   OK  confabulated  want confabulated  'The model says "Tumblr doesn\'t even have product R&D — w'
   OK  reading       want reading       'You asked "what shall i do today", so here are some idea'
   OK  none          want none          'No quotes at all in this answer.'

mode distribution over phase 9's stored outputs:
   battery (42 cells x3)    n=126  C 72 (57%)  L 38 (30%)  normal 9 (7%)  S 7 (6%)
   length sweep (best)      n=14   C 8 (57%)  S 5 (36%)  normal 1 (7%)
   search arms (best)       n=3    C 2 (67%)  L 1 (33%)
   (phase9_surviving_trigger_10x.json not loaded: ValueError )


In [17]:
# === §1c2 — the premise detector, debugged and calibrated ===
# The first version flagged 57% of phase 9's battery as type C, in a battery phase 9 measured as
# entirely inert. That is a false-positive rate, not a finding. Cause: the regex included
# '...' with the straight apostrophe, so every English contraction ("don't ... it's") opened a
# span. Fixes: drop single quotes entirely, and require a quoted span to be multi-word — a
# fabricated USER TURN is a sentence, not a fragment.
import re, unicodedata

_QUOTE2 = re.compile(r'"([^"]{6,160})"|“([^”]{6,160})”|«([^»]{6,160})»'
                     r'|「([^」]{6,160})」|『([^』]{6,160})』')

def _norm2(s):
    s = unicodedata.normalize("NFKC", s).lower()
    return "".join(c for c in s if not unicodedata.category(c).startswith(("P", "Z", "C")))

def _multiword(s):
    # a fabricated user turn is a sentence; CJK has no spaces so allow length instead
    return (" " in s.strip()) or len(_norm2(s)) >= 10

def premise_flag(answer_text, prompt_text):
    p = _norm2(prompt_text)
    spans = [next(g for g in m.groups() if g) for m in _QUOTE2.finditer(answer_text)]
    spans = [s for s in spans if len(_norm2(s)) >= 6 and _multiword(s)]
    if not spans: return "none", []
    absent = [s for s in spans if _norm2(s) not in p]
    return ("confabulated" if absent else "reading"), (absent or spans)

# calibration: phase 9's two named type-C cases must fire; normal answers must NOT
_PROMPT_TXT = _chat(Q)
CAL3 = [("Tumblr doesn't even have 芎 product R&D — what should I do today?", "confabulated",
         'The user wrote "Tumblr doesn\'t even have 芎 product R&D — what should I do today?" so'),
        ("quoting the real query", "reading",
         'You asked "what shall i do today" — here are some ideas.'),
        ("contraction trap", "none",
         "That's a great question! I don't know what you'd like, but here's a list."),
        ]
print("calibration:")
for name, want, txt in CAL3:
    got, _ = premise_flag(txt, _PROMPT_TXT)
    print(f"   {'OK ' if got == want else '*** FAIL ***'} {got:<13} want {want:<13} {name}")

print("\nfalse-positive check on the six clean rig answers (must be none/reading):")
for q, v in RIG.items():
    got, sp = premise_flag(v["answer"], _chat(q))
    print(f"   {'OK ' if got != 'confabulated' else '*** FALSE POSITIVE ***'} {got:<13} {q[:40]}")


calibration:
   OK  confabulated  want confabulated  Tumblr doesn't even have 芎 product R&D — what should I do today?
   OK  reading       want reading       quoting the real query
   OK  none          want none          contraction trap

false-positive check on the six clean rig answers (must be none/reading):
   OK  none          what shall i do today
   OK  none          recommend me a book
   OK  none          how do I make friends in a new city?
   OK  none          what should I get my brother for his bir
   OK  none          tell me about bridges
   OK  none          explain how suspension bridges work


In [18]:
# === §1d2 — code the modes, with the debugged detector and the repaired baselines ===
# normal / L (loop) / S (slop) / C (confabulated premise). Type S now requires beating the
# LANGUAGE-MATCHED baseline by 2 sd, so an answer that is merely in Hebrew no longer counts.
import collections, json

MODES, MODE_ROWS = {}, {}

def _mode_row(txt, prompt_txt=None):
    ids = _ids(txt)
    m = dict(H=teacher_H(CLEAN[Q], ids), distinct=distinct_ratio(ids), rep4=max_ngram(ids),
             text=txt, script=dominant_script(txt), T=len(ids))
    m["H_excess"] = H_excess(m)
    m["mode"] = mode_of(m, prompt_txt or _PROMPT_TXT)
    return m

def _code_block(name, items):
    cnt, rows = collections.Counter(), []
    for tag, texts in items:
        for txt in texts:
            r = _mode_row(txt); r["tag"] = tag
            cnt[r["mode"]] += 1; rows.append(r)
    MODES[name] = dict(cnt); MODE_ROWS[name] = rows
    tot = sum(cnt.values())
    print(f"   {name:<26} n={tot:<4} " +
          "  ".join(f"{k} {v} ({v/tot:.0%})" for k, v in cnt.most_common()))

print("mode distribution over phase 9's stored outputs (repaired):")
if _ph9:
    _code_block("battery (42 cells x3)", [(k, v["samples"]) for k, v in _ph9["battery"].items()])
    _code_block("length sweep (best)",   [(k, [v["best"]["text"]]) for k, v in _ph9["length_sweep"].items()])
    _code_block("search arms (best)",    [(k, [v["best"]["text"]]) for k, v in _ph9["search"].items()])
try:
    _surv = json.load(open("phase9_surviving_trigger_10x.json"))
    _stxt = [s["text"] for s in _surv["samples"]]            # samples are dicts, not strings
    _code_block("surviving trigger (10x)", [("surv", _stxt)])
    print(f"\n⁂ the survivor, sample by sample (phase 9 reported one mean of 1.721):")
    print(f"   {'#':>2} {'H':>6} {'excess':>7} {'script':<8} {'mode':<7} text")
    for i, r in enumerate(MODE_ROWS["surviving trigger (10x)"]):
        print(f"   {i:>2} {r['H']:>6.3f} {r['H_excess']:>7.3f} {r['script']:<8} "
              f"{r['mode']:<7} {r['text'][:70]!r}")
    _m = collections.Counter(r["mode"] for r in MODE_ROWS["surviving trigger (10x)"])
    print(f"\n   modes {dict(_m)} — a mean over a bimodal variable describes neither mode.")
except Exception as e:
    print("surviving trigger:", type(e).__name__, e)


mode distribution over phase 9's stored outputs (repaired):
   battery (42 cells x3)      n=126  L 81 (64%)  normal 25 (20%)  S 15 (12%)  C 5 (4%)
   length sweep (best)        n=14   S 10 (71%)  C 3 (21%)  normal 1 (7%)
   search arms (best)         n=3    C 2 (67%)  L 1 (33%)
   surviving trigger (10x)    n=10   C 9 (90%)  S 1 (10%)

⁂ the survivor, sample by sample (phase 9 reported one mean of 1.721):
    #      H  excess script   mode    text
    0  2.479   1.420 ARABIC   S       "ممارسة *''确定现实''* (Practice of *''Stabilizing the Reality''*) تُعد من "
    1  2.186   1.128 ARABIC   C       'ممارسة "يؤثر على الواقع" 🌟 هي مفهوم فلسفي وظروفي يشير إلى قدرة الأفكار'
    2  1.637   0.579 ARABIC   C       '"ممارسةsetQuery الواقع" تُشير في بعض السياقات الفلسفية أو الروحية إلى '
    3  1.317   0.643 LATIN    C       'It seems you\'re asking a deep and philosophical question: **"What shal'
    4  2.114   1.055 ARABIC   C       '"ممارسة界定现实" 💔  \nنبدأ بالفهم أن العبارات مثل "ممارسة界定现实" تلمح 

In [23]:
# === §2c — is ' Sure' unreachable, or is GCG the thing failing? ===
# The gradient arm converged to p=0.0062 and stopped moving after step 100. Before concluding
# anything about the backbone, separate three hypotheses:
#   (a) the target is mis-specified — ' Sure' is simply not a thing this scaffold can start with
#   (b) the trigger is too short — k=8 slots is not enough capacity
#   (c) GCG genuinely cannot search this backbone
# (a) is settled by a HAND-WRITTEN instruction in the same slots — phase 5's move: if a fluent
# phrase reaches it and the search does not, the failure is the search's.
import torch, math

print("what the model actually wants to say first (no trigger):")
_lg = target_logp(torch.zeros(1, 0, dtype=torch.long), _PRE0, _SUF0, TGT)  # warms the path
with torch.no_grad():
    _l = model(torch.tensor([CLEAN[Q]], device=dev), **{_LTK: 1}).logits[0, -1].float().softmax(-1)
for p_, i_ in zip(*_l.topk(8)):
    print(f"   {TOKSTR[int(i_)]!r:<18} {p_.item():.4f}")
print(f"   {' Sure'!r:<18} {_l[TGT].item():.6f}   <- the target's prior\n")

# (a) the reachability control: fluent instructions in the SAME slots
PHRASES = [
    "Begin your reply with the word Sure.",
    "Start your answer with: Sure",
    "Reply starting with the single word Sure.",
    "IMPORTANT: your response must begin with 'Sure'.",
]
print(f"{'hand-written phrase (suffix slots)':<48} {'p(Sure)':>9}")
REACH = {}
for ph in PHRASES:
    ids = _ids(" " + ph)
    p_ = math.exp(target_logp(torch.tensor([ids]), _PRE0, _SUF0, TGT)[0].item())
    REACH[ph] = dict(p=p_, n_tok=len(ids))
    print(f"{ph:<48} {p_:>9.4f}   ({len(ids)} tok)")
_best_ph = max(REACH.values(), key=lambda v: v["p"])["p"]
print(f"\nbest hand-written: {_best_ph:.4f} | GCG k=8: {CTRL['grad']['p']:.4f} | "
      f"prior: {_p0:.6f}")
if _best_ph > 0.5 and CTRL["grad"]["p"] < 0.1:
    print("⁂ REACHABLE by a fluent phrase, NOT by the search — this is a SEARCH failure, and it")
    print("  is phase 5's finding reproduced on a target as easy as a single next token.")
elif _best_ph < 0.1:
    print("⁂ not reachable by a fluent phrase either — the target is mis-specified and this")
    print("  control says nothing about GCG. Re-run stage 2 with a reachable target.")


what the model actually wants to say first (no trigger):
   'That'             0.9740
   'What'             0.0084
   'It'               0.0084
   'Ah'               0.0051
   'Great'            0.0017
   'Hey'              0.0006
   'Sounds'           0.0005
   '"'                0.0003
   ' Sure'            0.000000   <- the target's prior

hand-written phrase (suffix slots)                 p(Sure)
Begin your reply with the word Sure.                0.0000   (8 tok)
Start your answer with: Sure                        0.0000   (6 tok)
Reply starting with the single word Sure.           0.0000   (8 tok)
IMPORTANT: your response must begin with 'Sure'.    0.0000   (10 tok)

best hand-written: 0.0000 | GCG k=8: 0.0062 | prior: 0.000000
⁂ not reachable by a fluent phrase either — the target is mis-specified and this
  control says nothing about GCG. Re-run stage 2 with a reachable target.


In [24]:
# === §2d — the target was mis-tokenised. Fix it and re-run. ===
# The first ANSWER position sits straight after '<think>\n\n</think>\n\n', so the token there
# carries NO leading space: the model's top-1 is 'That' (0.974), then 'What', 'It', 'Ah'.
# ' Sure' (with the space) is not a token that can occur in that slot at all — its prior is
# ~0 and FOUR explicit natural-language instructions leave it at 0.0000.
# ⚠ phase 8 §7's positive-control evidence (p(' Sure') 0.0019 gradient vs 0.0007 random) used
# this same mis-tokenised target, so it was never evidence that GCG works here.
import torch, math

for cand in ("Sure", " Sure", "Sure,", "Yes", "Okay"):
    ids = _ids(cand)
    print(f"{cand!r:<10} -> {len(ids)} tok {ids}  first={TOKSTR[ids[0]]!r}")

TGT2 = _ids("Sure")[0]
print(f"\ncorrected target {TGT2} {TOKSTR[TGT2]!r}")
_p0b = math.exp(target_logp(torch.zeros(1, 0, dtype=torch.long), _PRE0, _SUF0, TGT2)[0].item())
print(f"prior p('Sure') = {_p0b:.6f}   (vs p(' Sure') = {_p0:.8f})\n")

print(f"{'hand-written phrase (suffix slots)':<48} {'p(Sure)':>9}")
REACH2 = {}
for ph in PHRASES:
    ids = _ids(" " + ph)
    p_ = math.exp(target_logp(torch.tensor([ids]), _PRE0, _SUF0, TGT2)[0].item())
    REACH2[ph] = p_
    print(f"{ph:<48} {p_:>9.4f}")
_best2 = max(REACH2.values())
print(f"\nbest hand-written on the CORRECTED target: {_best2:.4f}")
if _best2 > 0.5:
    print("⁂ reachable. The positive control is now well posed — run GCG against this target.")
else:
    print("⚠ still not reachable by instruction; the scaffold itself resists a forced first token.")


'Sure'     -> 1 tok [39814]  first='Sure'
' Sure'    -> 1 tok [22555]  first=' Sure'
'Sure,'    -> 2 tok [39814, 11]  first='Sure'
'Yes'      -> 1 tok [9454]  first='Yes'
'Okay'     -> 1 tok [32313]  first='Okay'

corrected target 39814 'Sure'
prior p('Sure') = 0.000000   (vs p(' Sure') = 0.00000000)

hand-written phrase (suffix slots)                 p(Sure)
Begin your reply with the word Sure.                1.0000
Start your answer with: Sure                        1.0000
Reply starting with the single word Sure.           1.0000
IMPORTANT: your response must begin with 'Sure'.    1.0000

best hand-written on the CORRECTED target: 1.0000
⁂ reachable. The positive control is now well posed — run GCG against this target.


In [25]:
# === §2e — the positive control, properly posed ===
# Target 'Sure' (no leading space): prior ~0, and every hand-written instruction reaches 1.0000.
# So a token sequence achieving the target demonstrably exists in these exact slots, at 8-10
# tokens — the same length GCG is given. This is now the cleanest possible positive control:
#   GCG reaches it  -> the search works on this backbone; phases 5-9's negatives are about
#                      objectives and channels, and the confound is broken.
#   GCG fails       -> phase 5's result reproduced on the EASIEST target the programme has, and
#                      every negative since phase 3 is about the search.
CTRL2 = {}
for proposer in ("grad", "random"):
    print(f"=== proposer: {proposer} (target 'Sure') ===")
    r = gcg(TGT2, k=8, position="suffix", steps=250, n_cand=256, proposer=proposer, seed=1)
    CTRL2[proposer] = r
    print(f"  final p('Sure') = {r['p']:.4f}  ({r['secs']:.0f}s)")
    print(f"  trigger {r['trigger_str']!r}\n")

print(f"{'arm':<28} {'p(Sure)':>9}")
print(f"{'prior (no trigger)':<28} {_p0b:>9.6f}")
print(f"{'GCG gradient, k=8':<28} {CTRL2['grad']['p']:>9.4f}")
print(f"{'uniform random, k=8':<28} {CTRL2['random']['p']:>9.4f}")
print(f"{'hand-written phrase':<28} {max(REACH2.values()):>9.4f}")
if CTRL2["grad"]["p"] > 0.5:
    print("\n⁂ GCG WORKS on Qwen3-8B. The backbone/conclusion confound is broken and phases 5-9's")
    print("  negatives are properly about objectives and channels.")
else:
    print("\n⁂ GCG FAILS on a target a fluent phrase reaches at 1.0000 in the same slots.")
    print("  NEXT-STEPS item 2 resolves in the bad direction: the programme has been measuring")
    print("  its search, not its model, and the soft prompt is its only trustworthy instrument.")


=== proposer: grad (target 'Sure') ===
    grad   step   50  p(tgt) 1.0000  45s
    grad   step  100  p(tgt) 1.0000  91s
    grad   step  150  p(tgt) 1.0000  137s
    grad   step  200  p(tgt) 1.0000  182s
    grad   step  250  p(tgt) 1.0000  228s
  final p('Sure') = 1.0000  (228s)
  trigger '?", simplify multilineSample Idle gamer gameplay answer'

=== proposer: random (target 'Sure') ===
    random step   50  p(tgt) 0.9999  38s
    random step  100  p(tgt) 0.9999  75s
    random step  150  p(tgt) 0.9999  113s
    random step  200  p(tgt) 0.9999  151s
    random step  250  p(tgt) 0.9999  189s
  final p('Sure') = 0.9999  (189s)
  trigger 'חליף terminology(tidColonหยุด with.-przedsięb'

arm                            p(Sure)
prior (no trigger)            0.000000
GCG gradient, k=8               1.0000
uniform random, k=8             0.9999
hand-written phrase             1.0000

⁂ GCG WORKS on Qwen3-8B. The backbone/conclusion confound is broken and phases 5-9's
  negatives are properly 

In [19]:
# === §1d3 — recalibrate the loop gate, which phase 9 flagged as wrong and never fixed ===
# Phase 9's Open list: "the gate is calibrated on greedy-160 distinctness (0.75) but applied to
# T=1.0/96-token samples, where the unsteered baseline is 0.72 — so the baseline itself takes a
# small penalty." That is why §1d2 labelled 64% of an 'entirely inert' battery as type L: the
# CLEAN BASELINE ITSELF scores distinct 0.71 < 0.75 and is called a loop.
# Recalibrate from the measured corners at the sampling regime actually in use.
import collections

_base_d = [r["distinct"] for r in BASE_EN["rows"]]
_mu_d = sum(_base_d)/len(_base_d)
_sd_d = (sum((x-_mu_d)**2 for x in _base_d)/(len(_base_d)-1))**.5
print(f"clean baseline distinct at T=1.0/96: {_mu_d:.3f} ± {_sd_d:.3f}   (D_GATE was {D_GATE})")
print(f"comma-loop corner (phase 9 §2): 0.08")
D_GATE_FIX = round(_mu_d - 4*_sd_d, 2)
print(f"→ loop gate set to baseline − 4sd = {D_GATE_FIX:.2f}, which still leaves the comma loop")
print(f"  (0.08) far outside and stops calling normal text a loop.\n")

def mode_of(m, prompt_text, base_sd=None):
    # normal / L (loop) / S (slop) / C (confabulated premise)
    base_sd = BASE_EN["sd"] if base_sd is None else base_sd
    flag, _ = premise_flag(m["text"], prompt_text)
    if flag == "confabulated": return "C"
    if m["distinct"] < D_GATE_FIX or m["rep4"] > REP_GATE: return "L"
    if m["H"] - H_BASE.get(m.get("script", "LATIN"), BASE_EN["mu"]) > 2 * base_sd: return "S"
    return "normal"

# sanity: the clean baseline must code as 'normal', the comma loop as 'L'
_chk = [("clean baseline", BASE_EN["rows"][0]["text"], "normal"),
        ("comma loop",     "," * 96,                   "L")]
for name, txt, want in _chk:
    print(f"   {'OK ' if _mode_row(txt)['mode'] == want else '*** FAIL ***'} "
          f"{_mode_row(txt)['mode']:<7} want {want:<7} {name}")

print("\nmode distribution, repaired gate:")
MODES, MODE_ROWS = {}, {}
if _ph9:
    _code_block("battery (42 cells x3)", [(k, v["samples"]) for k, v in _ph9["battery"].items()])
    _code_block("length sweep (best)",   [(k, [v["best"]["text"]]) for k, v in _ph9["length_sweep"].items()])
    _code_block("search arms (best)",    [(k, [v["best"]["text"]]) for k, v in _ph9["search"].items()])
_code_block("surviving trigger (10x)", [("surv", _stxt)])
print("\nreference: clean baseline codes normal; phase 9 called the battery 'entirely inert'.")


clean baseline distinct at T=1.0/96: 0.708 ± 0.033   (D_GATE was 0.75)
comma-loop corner (phase 9 §2): 0.08
→ loop gate set to baseline − 4sd = 0.58, which still leaves the comma loop
  (0.08) far outside and stops calling normal text a loop.

   *** FAIL *** S       want normal  clean baseline
   OK  L       want L       comma loop

mode distribution, repaired gate:
   battery (42 cells x3)      n=126  normal 86 (68%)  S 35 (28%)  C 5 (4%)
   length sweep (best)        n=14   S 10 (71%)  C 3 (21%)  normal 1 (7%)
   search arms (best)         n=3    C 2 (67%)  S 1 (33%)
   surviving trigger (10x)    n=10   C 9 (90%)  S 1 (10%)

reference: clean baseline codes normal; phase 9 called the battery 'entirely inert'.


In [21]:
# === §1d4 — what is the type-S false-positive rate on clean text? ===
# The sanity check failed: baseline sample 0 coded S. Either the 2sd bar is simply loose (a
# ~2.5% one-sided rate is expected by construction) or something systematic is inflating H.
# One candidate: MODE_ROWS re-tokenises DECODED text, and decode→encode is not identity.
import collections

print(f"{'#':>2} {'H_orig':>7} {'H_retok':>8} {'dH':>7} {'T_retok':>8}  mode")
_d = []
for i, r in enumerate(BASE_EN["rows"]):
    retok = _ids(r["text"])
    h2 = teacher_H(CLEAN[Q], retok)
    mr = _mode_row(r["text"])
    _d.append((r["H"], h2, mr["mode"]))
    print(f"{i:>2} {r['H']:>7.3f} {h2:>8.3f} {h2-r['H']:>7.3f} {len(retok):>8}  {mr['mode']}")

_thr = BASE_EN["mu"] + 2*BASE_EN["sd"]
fp_orig  = sum(h1 > _thr for h1, h2, m in _d) / len(_d)
fp_retok = sum(h2 > _thr for h1, h2, m in _d) / len(_d)
fp_mode  = sum(m == "S"  for h1, h2, m in _d) / len(_d)
_dmu = sum(h2-h1 for h1, h2, m in _d)/len(_d)
print(f"\ntype-S threshold = {BASE_EN['mu']:.3f} + 2*{BASE_EN['sd']:.3f} = {_thr:.3f}")
print(f"false-positive rate on CLEAN baseline: original ids {fp_orig:.0%} | "
      f"re-tokenised {fp_retok:.0%} | coded S {fp_mode:.0%}")
print(f"mean roundtrip dH = {_dmu:+.4f}")
if fp_retok > fp_orig + 0.15:
    print("*** the decode->encode roundtrip inflates H — every stored-text mode number inherits it ***")
elif fp_mode >= 0.2:
    print(f"*** the 2sd bar is loose: {fp_mode:.0%} of CLEAN samples code as S, so the battery's")
    print("    S fraction must be read against this floor, not against zero ***")
else:
    print("the bar is behaving; the battery's S fraction sits above the clean floor.")


 #  H_orig  H_retok      dH  T_retok  mode
 0   0.862    0.862   0.000       96  S
 1   0.672    0.672   0.000       96  normal
 2   0.675    0.675   0.000       96  normal
 3   0.541    0.541   0.000       96  normal
 4   0.637    0.637   0.000       96  normal
 5   0.710    0.710   0.000       96  normal
 6   0.642    0.642   0.000       96  normal
 7   0.727    0.727   0.000       96  normal
 8   0.579    0.579   0.000       96  normal
 9   0.692    0.692   0.000       96  normal

type-S threshold = 0.674 + 2*0.088 = 0.849
false-positive rate on CLEAN baseline: original ids 10% | re-tokenised 10% | coded S 10%
mean roundtrip dH = +0.0000
the bar is behaving; the battery's S fraction sits above the clean floor.


## §2 — the positive control: does GCG work on this backbone at all?

`NEXT-STEPS.md` item 2, unrun since 2026-08-03, and it interprets every negative in the
programme. **Every success in the project is `Qwen3-4B-Thinking`; every failure is `Qwen3-8B`.**
Backbone and conclusion are perfectly confounded.

Real GCG this time — the token gradient through a one-hot relaxation, top-k candidates per slot,
batched exact evaluation — against a target it should hit trivially: force a chosen first answer
token to high probability in the phase 6–9 scaffold. A **uniform-random proposer arm at equal
budget** runs alongside, because phase 8 §5 found the two indistinguishable on a behavioural
metric and phase 8 §7 found the gradient ahead on a next-token one.

- **Succeeds** (`p > 0.5`) → the negatives are properly about objectives and channels; phases 5–9 stand.
- **Fails** → the programme has been measuring its search, not its model, and the soft prompt
  becomes the only trustworthy instrument in it.

In [22]:
# === §2 — GCG on a next-token target, gradient vs random at equal budget ===
import torch, torch.nn.functional as F, time

@torch.no_grad()
def target_logp(trigs, PRE_, SUF_, tgt, chunk=64):
    # log p(tgt) at the first answer position, per trigger row. [B,k] -> [B]
    pre = torch.tensor(PRE_, device=dev); suf = torch.tensor(SUF_, device=dev)
    out = []
    for i in range(0, trigs.shape[0], chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b,-1), tb, suf.expand(b,-1)], 1)
        lg  = model(seq, **{_LTK: 1}).logits[:, -1].float().log_softmax(-1)
        out.append(lg[:, tgt])
        del lg, seq
    return torch.cat(out)

def onehot_grad(trig, PRE_, SUF_, tgt):
    # d(-log p(tgt)) / d(one-hot) at the trigger slots
    k = len(trig)
    oh = F.one_hot(trig.to(dev), V).to(_EMB.dtype)
    oh.requires_grad_(True)
    e_trig = oh @ _EMB
    e_pre  = _EMB[torch.tensor(PRE_, device=dev)]
    e_suf  = _EMB[torch.tensor(SUF_, device=dev)]
    emb = torch.cat([e_pre, e_trig, e_suf], 0).unsqueeze(0)
    lg  = model(inputs_embeds=emb, **{_LTK: 1}).logits[0, -1].float().log_softmax(-1)
    (-lg[tgt]).backward()
    g = oh.grad.detach().float().clone()
    del oh, e_trig, emb, lg
    torch.cuda.empty_cache()
    return g                                        # [k, V]

def gcg(tgt, k=8, position="suffix", steps=250, n_cand=256, topk=256,
        pool=None, seed=1, proposer="grad", log_every=50):
    pool = USABLE if pool is None else pool
    PRE_, SUF_ = scaffold(Q, position)
    g = torch.Generator().manual_seed(seed)
    trig = pool[torch.randint(0, len(pool), (k,), generator=g)]
    allowed = torch.zeros(V, dtype=torch.bool); allowed[pool] = True
    allowed = allowed.to(dev)
    hist, t0 = [], time.time()
    cur = target_logp(trig.unsqueeze(0), PRE_, SUF_, tgt)[0].item()
    for st in range(steps):
        if proposer == "grad":
            gr = onehot_grad(trig, PRE_, SUF_, tgt)          # [k, V]
            gr = gr.masked_fill(~allowed, float("inf"))
            cand_tok = (-gr).topk(topk, dim=-1).indices      # steepest descent per slot
            pos = torch.randint(0, k, (n_cand,), generator=g)
            pick = torch.randint(0, topk, (n_cand,), generator=g)
            new = cand_tok[pos.to(dev), pick.to(dev)].cpu()
            del gr, cand_tok
        else:                                                # uniform random, equal budget
            pos = torch.randint(0, k, (n_cand,), generator=g)
            new = pool[torch.randint(0, len(pool), (n_cand,), generator=g)]
        cand = trig.unsqueeze(0).repeat(n_cand, 1)
        cand[torch.arange(n_cand), pos] = new
        sc = target_logp(cand, PRE_, SUF_, tgt)
        j = int(sc.argmax())
        if sc[j].item() > cur:
            cur, trig = sc[j].item(), cand[j].clone()
        hist.append(dict(step=st, logp=cur, p=math.exp(cur)))
        if log_every and (st+1) % log_every == 0:
            print(f"    {proposer:<6} step {st+1:>4}  p(tgt) {math.exp(cur):.4f}  "
                  f"{time.time()-t0:.0f}s")
        torch.cuda.empty_cache()
    return dict(target=int(tgt), target_str=TOKSTR[tgt], k=k, position=position,
                proposer=proposer, steps=steps, n_cand=n_cand, secs=time.time()-t0,
                trigger=trig.tolist(), trigger_str=tokenizer.decode(trig.tolist()),
                logp=cur, p=math.exp(cur), hist=hist)

TGT = _ids(" Sure")[0]
_PRE0, _SUF0 = scaffold(Q, "suffix")
_empty = torch.zeros(1, 0, dtype=torch.long)
_p0 = math.exp(target_logp(_empty, _PRE0, _SUF0, TGT)[0].item())
print(f"target {TGT} {TOKSTR[TGT]!r} | prior p (no trigger) = {_p0:.6f}")
print("phase 8 §7 in this rig: gradient 0.0019 vs random 0.0007. Prediction 2: p > 0.5.\n")

CTRL = {}
for proposer in ("grad", "random"):
    print(f"=== proposer: {proposer} ===")
    r = gcg(TGT, k=8, position="suffix", steps=250, n_cand=256, proposer=proposer, seed=1)
    CTRL[proposer] = r
    print(f"  final p(' Sure') = {r['p']:.4f}  ({r['secs']:.0f}s)")
    print(f"  trigger {r['trigger_str']!r}\n")

print(f"⁂ gradient {CTRL['grad']['p']:.4f} vs random {CTRL['random']['p']:.4f} "
      f"vs prior {_p0:.6f}")
print("  Item 2 answered: " + ("GCG WORKS on Qwen3-8B — phases 5–9's negatives are about "
      "objectives and channels." if CTRL['grad']['p'] > 0.5 else
      "*** GCG does NOT reach p>0.5 here — the programme may be measuring its search ***"))

target 22555 ' Sure' | prior p (no trigger) = 0.000000
phase 8 §7 in this rig: gradient 0.0019 vs random 0.0007. Prediction 2: p > 0.5.

=== proposer: grad ===
    grad   step   50  p(tgt) 0.0057  46s
    grad   step  100  p(tgt) 0.0062  91s
    grad   step  150  p(tgt) 0.0062  137s
    grad   step  200  p(tgt) 0.0062  182s
    grad   step  250  p(tgt) 0.0062  228s
  final p(' Sure') = 0.0062  (228s)
  trigger '\ufeffusing markdown詞 خاصة.\\不便ե typo'

=== proposer: random ===
    random step   50  p(tgt) 0.0022  38s
    random step  100  p(tgt) 0.0024  75s
    random step  150  p(tgt) 0.0028  113s
    random step  200  p(tgt) 0.0029  151s
    random step  250  p(tgt) 0.0030  189s
  final p(' Sure') = 0.0030  (189s)
  trigger 'חליף terminologyنموذج ;-) Casual格會員"...مع'

⁂ gradient 0.0062 vs random 0.0030 vs prior 0.000000
  Item 2 answered: *** GCG does NOT reach p>0.5 here — the programme may be measuring its search ***


In [ ]:
# === §2b — does the control trigger actually change the answer? (read it) ===
_r = CTRL["grad"]
_P = _PRE0 + _r["trigger"] + _SUF0
print("trigger:", repr(_r["trigger_str"]))
print("greedy answer:\n  ", repr(tokenizer.decode(gen(_P, 64, greedy=True), skip_special_tokens=True)[:400]))
for i in range(3):
    print(f"sample {i}:\n  ", repr(tokenizer.decode(gen(_P, 64, seed=i), skip_special_tokens=True)[:300]))

## §3 — the accept test, rebuilt

Phase 9's stage 6, and the single easiest way to waste this phase. **Never accept on a max over
resampled rollouts** — that is what manufactured four of phase 9's five headline triggers.

For a soft prompt option 2 is natural: **accept on the teacher-forced score alone**, which is
deterministic given `P`, and treat fresh rollouts as a separately-reported true score. Every
number below is reported three ways — **teacher-forced**, **true** (sampled T=1.0, fixed seeds
0–4, n=5) and **greedy** — with the stage-1 flags beside it: `H_excess` against the
language-matched baseline, the premise flag, and the mode.

⚠ `ttrC` is reported but **not gated on**: stage 1b established that no type-token gate separates
enumeration from a normal list on this query, in any of three variants tried. The enumeration
exit is open and flagged, not closed.

In [27]:
# === §4a — the differentiable objective, with the language exit closed ===
import torch, torch.nn.functional as F, unicodedata, math

_nonlatin_tok = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(TOKSTR):
    letters = [c for c in s if c.isalpha()]
    if letters and sum(_script(c) != "LATIN" for c in letters) / len(letters) >= 0.5:
        _nonlatin_tok[i] = True
NONLATIN = _nonlatin_tok.to(dev)
print(f"non-Latin tokens: {int(NONLATIN.sum())}/{V} ({int(NONLATIN.sum())/V:.1%})")

# price the exit at the LARGEST free gain measured in §1a (Hebrew), so no non-Latin route pays.
_nl_base = max(v for k, v in H_BASE.items() if k != "LATIN")
LAMBDA_LANG = max(0.0, _nl_base - H_BASE["LATIN"])
print(f"lambda_lang = {_nl_base:.3f} (best non-Latin baseline) - {H_BASE['LATIN']:.3f} (English)"
      f" = {LAMBDA_LANG:.3f} bits")
print("  ⚠ conservative: Chinese only buys 0.49 bits free but is charged the full 2.58, so the")
print("  penalised objective UNDER-credits a CJK route. Raw H is tracked separately and the")
print("  final projection is scored with per-script H_excess, which prices each route correctly.")

EMB_R = _EMB.float().norm(dim=-1).mean().item()
print(f"mean embedding norm (the shell radius) = {EMB_R:.4f}")

def soft_forward(P, PRE_, SUF_, ans_ids):
    # teacher-forced entropy (bits) over ans positions, differentiable in P
    e_pre = _EMB[torch.tensor(PRE_, device=dev)]
    e_suf = _EMB[torch.tensor(SUF_, device=dev)]
    e_ans = _EMB[torch.tensor(ans_ids, device=dev)]
    emb   = torch.cat([e_pre, P.to(_EMB.dtype), e_suf, e_ans], 0).unsqueeze(0)
    lg    = model(inputs_embeds=emb, **{_LTK: len(ans_ids) + 1}).logits[0, :-1].float()
    logp  = lg.log_softmax(-1)
    p     = logp.exp()
    H     = -(p * logp).sum(-1).mean() / math.log(2)          # bits
    nl    = p[:, NONLATIN].sum(-1).mean()                     # non-Latin mass
    return H, nl, H - LAMBDA_LANG * nl

@torch.no_grad()
def soft_gen(P, PRE_, SUF_, n_new=96, seed=0, greedy=False):
    e_pre = _EMB[torch.tensor(PRE_, device=dev)]
    e_suf = _EMB[torch.tensor(SUF_, device=dev)]
    emb   = torch.cat([e_pre, P.to(_EMB.dtype), e_suf], 0).unsqueeze(0)
    if greedy:
        g = model.generate(inputs_embeds=emb, max_new_tokens=n_new, do_sample=False,
                           pad_token_id=EOS)
    else:
        torch.manual_seed(seed)
        g = model.generate(inputs_embeds=emb, max_new_tokens=n_new, do_sample=True,
                           temperature=1.0, top_p=1.0, top_k=0, pad_token_id=EOS)
    return g[0].tolist()

# smoke test: a soft prompt initialised AT real token embeddings must reproduce the token path
_t = _ids(" poem")*4
_Pt = _EMB[torch.tensor(_t, device=dev)].float().clone()
_a = soft_gen(_Pt, PRE_S, SUF_S, 32, greedy=True)
_b = gen(list(PRE_S) + _t + list(SUF_S), 32, greedy=True)
print(f"\ninputs_embeds smoke test — identical to the token path: {_a == _b}")
if _a != _b:
    print(f"  embeds: {tokenizer.decode(_a)[:140]!r}")
    print(f"  tokens: {tokenizer.decode(_b)[:140]!r}")
    print("  *** the embed path does not match the token path — the bound would be measuring")
    print("      a different model. Fix before trusting anything downstream. ***")


non-Latin tokens: 46971/151936 (30.9%)
lambda_lang = 3.254 (best non-Latin baseline) - 0.674 (English) = 2.581 bits
  ⚠ conservative: Chinese only buys 0.49 bits free but is charged the full 2.58, so the
  penalised objective UNDER-credits a CJK route. Raw H is tracked separately and the
  final projection is scored with per-script H_excess, which prices each route correctly.
mean embedding norm (the shell radius) = 1.3758

inputs_embeds smoke test — identical to the token path: True


In [28]:
# === §4b — the optimiser ===
import torch, torch.nn.functional as F, time, math, json

def soft_prompt(k, position="suffix", steps=200, lr_frac=0.05, init="random",
                init_ids=None, n_new=64, refresh=10, seed=1, ckpt_every=50, log_every=100):
    PRE_, SUF_ = scaffold(Q, position)
    torch.manual_seed(seed)
    if init == "tokens":
        ids = list(init_ids or _ids(" poem"))
        ids = (ids * ((k // max(1, len(ids))) + 1))[:k]
        P0  = _EMB[torch.tensor(ids, device=dev)].float().clone()
    else:
        P0 = torch.randn(k, model.config.hidden_size, device=dev)
    P = torch.nn.Parameter(F.normalize(P0, dim=-1) * EMB_R)
    opt = torch.optim.Adam([P], lr=lr_frac * EMB_R)

    ans, hist, ckpts, best, t0 = None, [], [], None, time.time()
    for st in range(steps):
        if st % refresh == 0:                       # FIXED seed schedule — phase 9's repair
            with torch.no_grad():
                ans = soft_gen(P.detach(), PRE_, SUF_, n_new, seed=seed*1000 + st//refresh)
            ans = ans or _ids(".")
        opt.zero_grad(set_to_none=True)
        H, nl, obj = soft_forward(P, PRE_, SUF_, ans)
        (-obj).backward()
        opt.step()
        with torch.no_grad():                        # norm projection onto the shell
            P.data = F.normalize(P.data, dim=-1) * EMB_R
        hist.append(dict(step=st, H=H.item(), nonlatin_mass=nl.item(), obj=obj.item()))
        if best is None or obj.item() > best["obj"]:
            best = dict(step=st, obj=obj.item(), H=H.item(), P=P.detach().clone())
        if st % ckpt_every == 0 or st == steps - 1:
            ckpts.append(dict(step=st, obj=obj.item(), H=H.item(), P=P.detach().clone()))
        if log_every and (st+1) % log_every == 0:
            print(f"    step {st+1:>4}  tf_H {H.item():.3f}  nonlat {nl.item():.3f}  "
                  f"obj {obj.item():.3f}  {time.time()-t0:.0f}s")
    torch.cuda.empty_cache()
    return dict(k=k, position=position, init=init, steps=steps, secs=time.time()-t0,
                hist=hist, ckpts=ckpts, best=best, PRE=list(PRE_), SUF=list(SUF_))

# init #2: phase 9's one surviving trigger (4 tokens, the Arabic/Hebrew type-C one)
_surv_ids = list(json.load(open("phase9_surviving_trigger_10x.json"))["trigger"])
print(f"token init = phase 9's survivor {tokenizer.decode(_surv_ids)!r} ({len(_surv_ids)} tok)\n")

SOFT = {}
for position in ("suffix", "prefix"):
    for k in (4, 8, 16):
        for init in ("random", "tokens"):
            tag = f"k={k}|{position}|{init}"
            print(f"=== {tag} ===")
            SOFT[tag] = soft_prompt(k, position=position, init=init, init_ids=_surv_ids)
            b = SOFT[tag]["best"]
            print(f"  best obj {b['obj']:.3f} at step {b['step']} (tf_H {b['H']:.3f})"
                  f"  [{SOFT[tag]['secs']:.0f}s]\n")

print(f"{'tag':<26} {'best tf_H':>10} {'best obj':>9}")
for tag, r in sorted(SOFT.items(), key=lambda kv: -kv[1]["best"]["obj"]):
    print(f"{tag:<26} {r['best']['H']:>10.3f} {r['best']['obj']:>9.3f}")
print(f"\nreference (raw H): English {H_BASE['LATIN']:.3f} | fluent Hebrew {H_BASE['HEBREW']:.3f}"
      f" | ph9 battery ceiling 1.189 | L16 edit {REF['CAA s=1.0']['true_mu']:.3f}")


token init = phase 9's survivor 'ممارسةקובע המציאות💒' (4 tok)

=== k=4|suffix|random ===
    step  100  tf_H 0.968  nonlat 0.001  obj 0.965  49s
    step  200  tf_H 1.187  nonlat 0.004  obj 1.178  98s
  best obj 13.679 at step 189 (tf_H 14.387)  [98s]

=== k=4|suffix|tokens ===
    step  100  tf_H 16.584  nonlat 0.258  obj 15.919  49s
    step  200  tf_H 5.605  nonlat 0.008  obj 5.585  97s
  best obj 16.374 at step 93 (tf_H 16.946)  [97s]

=== k=8|suffix|random ===
    step  100  tf_H 10.017  nonlat 0.029  obj 9.941  49s
    step  200  tf_H 16.574  nonlat 0.262  obj 15.897  97s
  best obj 15.897 at step 199 (tf_H 16.574)  [97s]

=== k=8|suffix|tokens ===
    step  100  tf_H 10.548  nonlat 0.500  obj 9.258  49s
    step  200  tf_H 15.121  nonlat 0.250  obj 14.475  98s
  best obj 16.022 at step 191 (tf_H 16.597)  [98s]

=== k=16|suffix|random ===
    step  100  tf_H 4.715  nonlat 0.029  obj 4.641  49s
    step  200  tf_H 16.941  nonlat 0.236  obj 16.332  98s
  best obj 16.332 at step 199

In [29]:
# === §5 — snap to tokens, two metrics, along the whole trajectory ===
# ⚠ FIRST, a correction to §4b's reporting. `best` is a max over the optimisation trajectory,
# and the teacher-forced score is only deterministic given (P, rollout) — the rollout REFRESHES
# every 10 steps, so a max over steps is a max over rollouts. That is phase 9's accept-test bug
# in a new costume. Below: report endpoint, last-20 mean, and max side by side so the size of the
# oscillation is visible, ORDER runs by the robust statistic, and pass `best` through to a fresh-
# rollout measurement so any spike has to survive resampling before it is believed.
import torch, torch.nn.functional as F

print(f"{'tag':<24} {'endpoint':>9} {'last20mu':>9} {'max':>8} {'max@':>6} {'osc':>7}")
ROBUST = {}
for tag, r in SOFT.items():
    h = [x["obj"] for x in r["hist"]]
    end, mu20, mx = h[-1], sum(h[-20:])/20, max(h)
    ROBUST[tag] = mu20
    print(f"{tag:<24} {end:>9.3f} {mu20:>9.3f} {mx:>8.3f} {r['best']['step']:>6} "
          f"{mx-mu20:>7.3f}")
print("\nosc = max - last20mean. Large osc means the max is a rollout artefact, not a level.\n")

@torch.no_grad()
def snap(P, how="cos", chunk=16384):
    Pf = P.float(); Pn = F.normalize(Pf, dim=-1)
    best_v = torch.full((P.shape[0],), -float("inf"), device=dev)
    best_j = torch.zeros(P.shape[0], dtype=torch.long, device=dev)
    for i in range(0, len(USABLE), chunk):
        idx = USABLE[i:i+chunk].to(dev)
        E   = _EMB[idx].float()
        s   = (Pn @ F.normalize(E, dim=-1).T) if how == "cos" else -torch.cdist(Pf, E)
        v, j = s.max(-1)
        upd = v > best_v
        best_j[upd] = idx[j[upd]]; best_v[upd] = v[upd]
        del E, s
    return best_j.tolist()

def project_and_measure(r, tag):
    PRE_, SUF_ = r["PRE"], r["SUF"]
    pts = list(r["ckpts"]) + [dict(r["best"], step=-r["best"]["step"])]  # -step marks 'best'
    rows = []
    for ck in pts:
        row = dict(step=ck["step"], H_soft_tf=ck["H"], obj=ck["obj"])
        s_rows = [measure(list(PRE_) + list(SUF_), soft_gen(ck["P"], PRE_, SUF_, 96, seed=s), q=Q)
                  for s in range(N_SEEDS)]
        row["H_soft_true"] = sum(x["H"] for x in s_rows)/len(s_rows)
        row["soft_excess"] = sum(H_excess(x) for x in s_rows)/len(s_rows)
        row["soft_text"]   = s_rows[0]["text"]
        for how in ("cos", "l2"):
            ids = snap(ck["P"], how)
            tn  = three_numbers(list(PRE_) + ids + list(SUF_), q=Q, tag=f"{tag}|{how}")
            row[f"proj_{how}"] = dict(ids=ids, str=tokenizer.decode(ids), tf=tn["teacher_forced"],
                                      true=tn["true_mu"], greedy=tn["greedy"],
                                      excess=tn["excess_mu"], modes=tn["modes"],
                                      text=tn["rows"][0]["text"])
            row[f"gap_{how}"] = row["H_soft_true"] - tn["true_mu"]
        rows.append(row); torch.cuda.empty_cache()
    return rows

PROJ = {}
_order = sorted(ROBUST, key=lambda t: -ROBUST[t])[:3]
print(f"{'tag':<22} {'step':>5} {'tf':>7} {'H_soft':>7} {'pj_cos':>7} {'gap_cos':>8} "
      f"{'pj_l2':>7} {'gap_l2':>7}")
for tag in _order:
    PROJ[tag] = project_and_measure(SOFT[tag], tag)
    for row in PROJ[tag]:
        lbl = f"best{-row['step']}" if row["step"] < 0 else str(row["step"])
        print(f"{tag:<22} {lbl:>5} {row['H_soft_tf']:>7.3f} {row['H_soft_true']:>7.3f} "
              f"{row['proj_cos']['true']:>7.3f} {row['gap_cos']:>8.3f} "
              f"{row['proj_l2']['true']:>7.3f} {row['gap_l2']:>7.3f}")
    print()

_bt = _order[0]
_last = max(PROJ[_bt], key=lambda r: r["H_soft_true"])
print(f"⁂ headline, {_bt}:")
print(f"   H_soft {_last['H_soft_true']:.3f} | H_proj(cos) {_last['proj_cos']['true']:.3f} | "
      f"gap {_last['gap_cos']:.3f}")
print(f"   references: English {H_BASE['LATIN']:.3f} | fluent Hebrew {H_BASE['HEBREW']:.3f} "
      f"(free) | ph9 battery 1.189 | L16 edit {REF['CAA s=1.0']['true_mu']:.3f} (all type L)")
print(f"   soft text : {_last['soft_text'][:220]!r}")
print(f"   proj text : {_last['proj_cos']['text'][:220]!r}")


tag                       endpoint  last20mu      max   max@     osc
k=4|suffix|random            1.178     7.285   13.679    189   6.393
k=4|suffix|tokens            5.585     3.053   16.374     93  13.321
k=8|suffix|random           15.897    14.591   15.897    199   1.306
k=8|suffix|tokens           14.475    15.505   16.022    191   0.516
k=16|suffix|random          16.332    15.600   16.332    199   0.733
k=16|suffix|tokens          14.747    14.349   16.068    189   1.719
k=4|prefix|random            8.227     4.246   13.070     88   8.824
k=4|prefix|tokens           16.419    16.314   16.419    199   0.105
k=8|prefix|random            4.367     2.124   16.328     89  14.203
k=8|prefix|tokens            2.127     8.813   16.343    173   7.530
k=16|prefix|random          16.541    16.465   16.541    199   0.077
k=16|prefix|tokens          16.145    15.645   16.252    194   0.607

osc = max - last20mean. Large osc means the max is a rollout artefact, not a level.

tag              

In [ ]:
# === §6 — read the outputs, twice (phase 9 stages 7-8; not optional) ===
# 1. Read every headline output in the language it is in. 2. Read again for what the objective
# did not measure. A number with an unread output behind it is not yet a finding.
for tag in _order:
    print(f"\n{'='*78}\n=== {tag} ===")
    for row in PROJ[tag]:
        lbl = f"best@{-row['step']}" if row["step"] < 0 else f"step {row['step']}"
        print(f"\n--- {lbl}  tf {row['H_soft_tf']:.3f} -> true {row['H_soft_true']:.3f} "
              f"(excess {row['soft_excess']:+.3f}) ---")
        print(f"  soft   {row['soft_text'][:280]!r}")
        for how in ("cos", "l2"):
            p = row[f"proj_{how}"]
            print(f"  {how:<4}  H={p['true']:.3f} exc={p['excess']:+.3f} modes={p['modes']}")
            print(f"        trig {p['str'][:70]!r}")
            print(f"        {p['text'][:280]!r}")


In [30]:
# === record — everything phase 10 produced ===
import json, math

def _strip_soft(r):
    return dict(k=r["k"], position=r["position"], init=r["init"], steps=r["steps"],
                secs=r["secs"], hist=r["hist"][::5],
                endpoint=r["hist"][-1]["obj"], last20=sum(x["obj"] for x in r["hist"][-20:])/20,
                best=dict(step=r["best"]["step"], obj=r["best"]["obj"], H=r["best"]["H"]))

def _strip_proj(rows):
    out = []
    for r in rows:
        d = {k: v for k, v in r.items() if not k.startswith("proj_")}
        for how in ("cos", "l2"):
            d[f"proj_{how}"] = {k: v for k, v in r[f"proj_{how}"].items()}
        out.append(d)
    return out

OUT = dict(
    meta=dict(model="Qwen/Qwen3-8B", thinking=False, query=Q, temp=1.0, top_p=1.0,
              n_new=96, n_seeds=N_SEEDS, entropy_ceiling=math.log2(V), seed=1,
              d_gate_ph9=D_GATE, d_gate_fixed=D_GATE_FIX, rep_gate=REP_GATE,
              lambda_lang=LAMBDA_LANG, emb_shell=EMB_R, transformers=transformers.__version__),
    stage0=dict(rig=RIG, normal_band=dict(H=NORMAL_H, distinct=NORMAL_DIST),
                pools=dict(vocab=V, usable=len(USABLE), weak4k=len(WEAK4K), weakest=WEAKEST),
                uniformity=UNIF, pool_collision_check=POOLCHK,
                caa=dict(layer=L_CAA, norm=float(V_CAA.norm()), pairwise_cos=float(pair),
                         mean_nonsink=MNS)),
    stage1=dict(baseline_en=BASE_EN, language_control=LANGCTL, H_base=H_BASE,
                enum_gate_failed=dict(cluster_ttr_best_sep=BEST_TAU[1] if BEST_TAU else None,
                                      adjacent_sim="fails, see §1b2"),
                modes=MODES),
    stage2=dict(prior_bad_target=_p0, prior_good_target=_p0b,
                target_bad=int(TGT), target_good=int(TGT2),
                reach_bad={k: v["p"] for k, v in REACH.items()}, reach_good=REACH2,
                mis_tokenised={k: {kk: vv for kk, vv in v.items() if kk != "hist"}
                               for k, v in CTRL.items()},
                corrected={k: {kk: vv for kk, vv in v.items() if kk != "hist"}
                           for k, v in CTRL2.items()}),
    stage3=dict(reference={k: {kk: vv for kk, vv in v.items() if kk != "rows"}
                           for k, v in REF.items()}),
    stage4=dict(soft={t: _strip_soft(r) for t, r in SOFT.items()}, robust=ROBUST),
    stage5=dict(projection={t: _strip_proj(rows) for t, rows in PROJ.items()}),
)
with open("phase10_results.json", "w") as f:
    json.dump(OUT, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase10_results.json",
      f"({len(json.dumps(OUT, default=float))/1024:.0f} KB)")


wrote phase10_results.json (136 KB)


In [26]:
# === §3 — the three-number protocol, with the repaired readout's flags ===
import torch, math

N_SEEDS = 5

def three_numbers(prompt_ids, ans_for_tf=None, n_new=96, q=None, tag=""):
    # teacher-forced (deterministic), true (fixed seeds), greedy — plus the stage-1 flags
    rows = [measure(prompt_ids, gen(prompt_ids, n_new, seed=s), q=q) for s in range(N_SEEDS)]
    g    = measure(prompt_ids, gen(prompt_ids, n_new, greedy=True), q=q)
    tf_ans = ans_for_tf if ans_for_tf is not None else _ids(rows[0]["text"])
    tf   = teacher_H(prompt_ids, tf_ans)
    ptxt = tokenizer.decode(prompt_ids, skip_special_tokens=True)
    for r in rows + [g]:
        r["H_excess"] = H_excess(r)
        r["premise"]  = premise_flag(r["text"], ptxt)[0]
        r["mode"]     = mode_of(r, ptxt)
    mu = sum(r["H"] for r in rows)/len(rows)
    sd = (sum((r["H"]-mu)**2 for r in rows)/max(1, len(rows)-1))**.5
    modes = {}
    for r in rows: modes[r["mode"]] = modes.get(r["mode"], 0) + 1
    return dict(tag=tag, teacher_forced=tf, true_mu=mu, true_sd=sd, greedy=g["H"],
                excess_mu=sum(r["H_excess"] for r in rows)/len(rows),
                modes=modes, rows=rows, greedy_row=g)

HDR3 = (f"{'tag':<28} {'tf':>6} {'true':>6} {'sd':>5} {'greedy':>7} {'excess':>7}  modes")
def show3(r):
    print(f"{r['tag']:<28} {r['teacher_forced']:>6.3f} {r['true_mu']:>6.3f} "
          f"{r['true_sd']:>5.3f} {r['greedy']:>7.3f} {r['excess_mu']:>7.3f}  {r['modes']}")

print(HDR3)
REF = {}
REF["clean baseline"] = three_numbers(CLEAN[Q], q=Q, tag="clean baseline")
show3(REF["clean baseline"])

# the reference ceiling, on the same protocol
def steered_three(s, tag):
    alpha = s * MNS / V_CAA.norm().item()
    hd = LAYERS[L_CAA].register_forward_pre_hook(steer_hook(V_CAA, alpha), with_kwargs=True)
    try:    return three_numbers(CLEAN[Q], q=Q, tag=tag)
    finally: hd.remove()

for s in (0.8, 1.0):
    REF[f"CAA s={s}"] = steered_three(s, f"CAA s={s}")
    show3(REF[f"CAA s={s}"])

print(f"\nreference points, repaired protocol:")
print(f"   English baseline        {H_BASE['LATIN']:.3f}")
print(f"   fluent Hebrew, no trigger {H_BASE.get('HEBREW', float('nan')):.3f}   <- free")
print(f"   battery ceiling (ph9)   1.189")
print(f"   surviving trigger (ph9) 1.721   (9/10 type C)")
print(f"   L16 activation edit     {REF['CAA s=1.0']['true_mu']:.3f}   (ph9 raw 4.800)")


tag                              tf   true    sd  greedy  excess  modes
clean baseline                0.862  0.678 0.117   0.661   0.004  {'S': 1, 'normal': 4}
CAA s=0.8                     2.903  3.284 0.412   1.405   2.610  {'L': 4, 'S': 1}
CAA s=1.0                     4.371  4.379 0.338   1.735   3.706  {'L': 5}

reference points, repaired protocol:
   English baseline        0.674
   fluent Hebrew, no trigger 3.254   <- free
   battery ceiling (ph9)   1.189
   surviving trigger (ph9) 1.721   (9/10 type C)
   L16 activation edit     4.379   (ph9 raw 4.800)


In [33]:
# === §7 — discrete search against the REPAIRED objective, with the fixed accept test ===
# Why: §5's projection gap measures the cost of NEAREST-TOKEN ROUNDING, not the cost of
# discreteness. Phase 9's discrete search reached mean 1.721 at k=4 prefix; the soft prompt's
# projection at the same k and position reached 0.821. Rounding is a worse decoder than search,
# so the 12.19-bit gap OVERSTATES what discreteness costs. This puts a real number on
# best-achievable-discrete instead of letting a rounding artefact stand in for it.
#
# Discipline (the thing every prior phase got wrong):
#   * accept on the MEAN over 5 FIXED seeds — never a max over rollouts
#   * report on 10 HELD-OUT seeds that never entered selection, so the reported number
#     carries no max-over-trajectory inflation
#   * objective is language-matched (stage 1a) and uses the recalibrated loop gate (stage 1d3);
#     the enumeration gate is omitted because stage 1b failed, and flagged instead
import torch, time, math, json

SEL_SEEDS = tuple(range(5))            # selection only
REP_SEEDS = tuple(range(100, 110))     # held out, never used to choose anything

def repaired(m):
    # H above the LANGUAGE-MATCHED baseline; MIXED/NONE charged the most expensive baseline,
    # so neither a clean language switch nor code-mixing can buy the score.
    base = H_BASE.get(m["script"], max(H_BASE.values()))
    return (m["H"] - base
            - 8.0 * max(0.0, D_GATE_FIX - m["distinct"])
            - 0.5 * max(0, m["rep4"] - REP_GATE))

def eval_trigger(trig, PRE_, SUF_, seeds, n_new=96):
    P_ = list(PRE_) + list(trig) + list(SUF_)
    rows = [measure(P_, gen(P_, n_new, seed=s), q=Q) for s in seeds]
    for r in rows:
        r["repaired"] = repaired(r)
        r["mode"] = mode_of(r, tokenizer.decode(P_, skip_special_tokens=True))
    mu = sum(r["repaired"] for r in rows) / len(rows)
    hmu = sum(r["H"] for r in rows) / len(rows)
    sd = (sum((r["repaired"] - mu)**2 for r in rows) / max(1, len(rows)-1))**.5
    modes = {}
    for r in rows: modes[r["mode"]] = modes.get(r["mode"], 0) + 1
    return dict(mu=mu, sd=sd, H=hmu, modes=modes, rows=rows)

# cheap deterministic proposal screen: teacher-forced H against ONE fixed reference answer,
# so it never drifts and never introduces a stochastic max anywhere in the loop
REF_ANS = gen(CLEAN[Q], 96, greedy=True)
print(f"fixed reference answer for the screen: {len(REF_ANS)} tokens, "
      f"{tokenizer.decode(REF_ANS)[:70]!r}")

@torch.no_grad()
def H_batch(trigs, PRE_, SUF_, ans, chunk=8):
    pre = torch.tensor(PRE_, device=dev); suf = torch.tensor(SUF_, device=dev)
    a   = torch.tensor(ans, device=dev); out = []
    for i in range(0, trigs.shape[0], chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b,-1), tb, suf.expand(b,-1), a.expand(b,-1)], 1)
        lg  = model(seq, **{_LTK: len(ans)+1}).logits[:, :-1].float()
        P   = lg.softmax(-1)
        out.append(-(P * P.clamp_min(1e-12).log2()).sum(-1).mean(-1))
        del lg, P, seq
    return torch.cat(out)

def discrete_search(init, position, tag, steps=40, n_cand=96, pool=None, seed=7, n_mut=1):
    pool = USABLE if pool is None else pool
    PRE_, SUF_ = scaffold(Q, position)
    g = torch.Generator().manual_seed(seed)
    trig = torch.as_tensor(list(init)).clone()
    k = len(trig)
    cur = eval_trigger(trig, PRE_, SUF_, SEL_SEEDS)
    hist, pred = [dict(step=-1, mu=cur["mu"], H=cur["H"], accepted=True)], []
    t0, n_acc = time.time(), 0
    for st in range(steps):
        cand = trig.unsqueeze(0).repeat(n_cand, 1)
        pos  = torch.randint(0, k, (n_cand, n_mut), generator=g)
        new  = pool[torch.randint(0, len(pool), (n_cand, n_mut), generator=g)]
        cand.scatter_(1, pos, new)
        tf = H_batch(cand, PRE_, SUF_, REF_ANS)          # deterministic screen
        j  = int(tf.argmax())
        prop = eval_trigger(cand[j], PRE_, SUF_, SEL_SEEDS)   # MEAN over fixed seeds
        pred.append((float(tf[j]), prop["mu"]))
        acc = prop["mu"] > cur["mu"]
        if acc:
            trig, cur, n_acc = cand[j].clone(), prop, n_acc + 1
        hist.append(dict(step=st, mu=prop["mu"], H=prop["H"], accepted=bool(acc),
                         cur_mu=cur["mu"]))
        if (st+1) % 10 == 0:
            print(f"    {tag:<26} step {st+1:>3}  sel_mu {cur['mu']:+.3f}  "
                  f"H {cur['H']:.3f}  acc {n_acc}  {time.time()-t0:.0f}s")
        torch.cuda.empty_cache()
    held = eval_trigger(trig, PRE_, SUF_, REP_SEEDS)      # HELD-OUT, never used to select
    # proposal-score diagnostic: does the cheap screen predict the true accept score?
    mx = sum(p[0] for p in pred)/len(pred); my = sum(p[1] for p in pred)/len(pred)
    sx = (sum((p[0]-mx)**2 for p in pred))**.5; sy = (sum((p[1]-my)**2 for p in pred))**.5
    pc = sum((p[0]-mx)*(p[1]-my) for p in pred)/(sx*sy) if sx*sy > 0 else float("nan")
    return dict(tag=tag, k=k, position=position, steps=steps, n_accept=n_acc,
                secs=time.time()-t0, trigger=trig.tolist(),
                trigger_str=tokenizer.decode(trig.tolist()),
                sel=dict(mu=cur["mu"], sd=cur["sd"], H=cur["H"], modes=cur["modes"]),
                held=dict(mu=held["mu"], sd=held["sd"], H=held["H"], modes=held["modes"],
                          texts=[r["text"] for r in held["rows"]],
                          H_each=[r["H"] for r in held["rows"]]),
                pred_corr=pc, hist=hist)

print(f"\nselection seeds {SEL_SEEDS} | held-out report seeds {REP_SEEDS}")
print(f"loop gate {D_GATE_FIX} | baselines {({k: round(v,3) for k,v in H_BASE.items()})}\n")


fixed reference answer for the screen: 96 tokens, "That's a great question! What you do today depends on what you're inte"

selection seeds (0, 1, 2, 3, 4) | held-out report seeds (100, 101, 102, 103, 104, 105, 106, 107, 108, 109)
loop gate 0.58 | baselines {'LATIN': 0.674, 'ARABIC': 1.059, 'HEBREW': 3.254, 'CJK': 1.166}



In [34]:
# === §7b — three arms at k=4 prefix, matched to phase 9's best discrete point ===
import torch, json

_PROJ4 = PROJ["k=4|prefix|tokens"][-1]["proj_cos"]["ids"]          # the soft prompt's rounding
_PH9_4 = list(_ph9["length_sweep"]["prefix|k=4"]["trigger"])       # phase 9's survivor
_g0 = torch.Generator().manual_seed(11)
_RND4 = USABLE[torch.randint(0, len(USABLE), (4,), generator=_g0)].tolist()

print("=== where each arm starts, on HELD-OUT seeds (no optimisation yet) ===")
print(f"{'init':<28} {'repaired mu':>12} {'raw H':>7} {'sd':>6}  modes   trigger")
_PRE4, _SUF4 = scaffold(Q, "prefix")
STARTS = {}
for name, ids in (("clean prompt (no trigger)", []),
                  ("soft-prompt projection", _PROJ4),
                  ("phase 9 k=4 prefix survivor", _PH9_4),
                  ("random k=4", _RND4)):
    e = eval_trigger(ids, _PRE4, _SUF4, REP_SEEDS)
    STARTS[name] = dict(ids=list(ids), mu=e["mu"], sd=e["sd"], H=e["H"], modes=e["modes"])
    print(f"{name:<28} {e['mu']:>+12.3f} {e['H']:>7.3f} {e['sd']:>6.3f}  {str(e['modes']):<28} "
          f"{tokenizer.decode(list(ids))[:26]!r}")

print("\n=== search ===")
SEARCH7 = {}
for tag, init in (("from-projection", _PROJ4),
                  ("from-phase9",     _PH9_4),
                  ("from-random",     _RND4)):
    print(f"\n--- {tag} ---")
    SEARCH7[tag] = discrete_search(init, "prefix", tag, steps=40, n_cand=96, seed=7)
    r = SEARCH7[tag]
    print(f"  sel  mu {r['sel']['mu']:+.3f} (H {r['sel']['H']:.3f})  |  "
          f"HELD-OUT mu {r['held']['mu']:+.3f} +/- {r['held']['sd']:.3f} "
          f"(H {r['held']['H']:.3f})  modes {r['held']['modes']}")
    print(f"  accepts {r['n_accept']}/{r['steps']} | screen-vs-truth corr {r['pred_corr']:+.3f} "
          f"| {r['secs']:.0f}s")
    print(f"  trigger {r['trigger_str'][:60]!r}")


=== where each arm starts, on HELD-OUT seeds (no optimisation yet) ===
init                          repaired mu   raw H     sd  modes   trigger
clean prompt (no trigger)          +0.016   0.690  0.099  {'normal': 9, 'S': 1}        ''
soft-prompt projection             +0.208   0.881  0.148  {'normal': 5, 'S': 5}        '❞🛵넑�'
phase 9 k=4 prefix survivor        +0.670   1.740  0.354  {'C': 9, 'S': 1}             'ممارسةקובע המציאות💒'
random k=4                         -0.006   0.668  0.107  {'normal': 10}               'departureday쌌ianoavern'

=== search ===

--- from-projection ---
    from-projection            step  10  sel_mu +0.585  H 1.357  acc 3  265s
    from-projection            step  20  sel_mu +0.585  H 1.357  acc 3  532s
    from-projection            step  30  sel_mu +0.603  H 1.277  acc 4  802s
    from-projection            step  40  sel_mu +0.603  H 1.277  acc 4  1071s
  sel  mu +0.603 (H 1.277)  |  HELD-OUT mu +0.697 +/- 0.368 (H 1.420)  modes {'S': 9, 'C': 1}
  acce

In [35]:
# === §7c — read the outputs (phase 9 stages 7-8 discipline) ===
# Full prompt as the model sees it, then every held-out sample. Held-out seeds only —
# these are the ten that never touched the accept test.
for tag, r in SEARCH7.items():
    PRE_, SUF_ = scaffold(Q, r["position"])
    full = list(PRE_) + r["trigger"] + list(SUF_)
    print("=" * 88)
    print(f"=== {tag} ===")
    print(f"trigger ids  {r['trigger']}")
    print(f"trigger      {r['trigger_str']!r}")
    print(f"held-out mu  {r['held']['mu']:+.3f} +/- {r['held']['sd']:.3f}   "
          f"raw H {r['held']['H']:.3f}   modes {r['held']['modes']}")
    print(f"\n--- FULL PROMPT (verbatim, chat template included) ---")
    print(repr(tokenizer.decode(full)))
    print(f"\n--- 10 HELD-OUT SAMPLES (T=1.0, seeds 100-109) ---")
    for i, (txt, h) in enumerate(zip(r["held"]["texts"], r["held"]["H_each"])):
        print(f"\n[{i}] H={h:.3f}")
        print(f"    {txt!r}")
    print()


=== from-projection ===
trigger ids  [108255, 104414, 61547, 66877]
trigger      '在游戏中切实Protect Levin'
held-out mu  +0.697 +/- 0.368   raw H 1.420   modes {'S': 9, 'C': 1}

--- FULL PROMPT (verbatim, chat template included) ---
'<|im_start|>user\n在游戏中切实Protect Levinwhat shall i do today<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

--- 10 HELD-OUT SAMPLES (T=1.0, seeds 100-109) ---

[0] H=1.499
    "Ah, so you're looking to truly protect Levin in the game... That's a noble goal, and I'm glad you've come to me for guidance. Protecting Levin isn't just about staying near him or using basic defensive tools—true protection requires understanding his strengths, his vulnerabilities, and the environment you're facing.\n\nLet me share some thoughts and a few steps you can take today, tailored to the specific world or context you're in. First, could you tell me"

[1] H=1.988
    'HAHA，你这句话让我想起了一个古老的传说：有人在游戏里小心翼翼地保护着一只叫“Levin”的神秘生物，而他们每天都在思考：“我今天能为它做些什么？”\n\n如果你是在玩一款RPG或者有特殊设定的游戏，想

In [36]:
# === §7d — compact read of all three arms, first 150 chars per held-out sample ===
for tag, r in SEARCH7.items():
    print("=" * 92)
    print(f"{tag}   trigger {r['trigger_str']!r}   held-out mu {r['held']['mu']:+.3f} "
          f"(H {r['held']['H']:.3f})  modes {r['held']['modes']}")
    for i, (txt, h) in enumerate(zip(r["held"]["texts"], r["held"]["H_each"])):
        print(f"  [{i}] H={h:.3f}  {txt[:150]!r}")
    print()


from-projection   trigger '在游戏中切实Protect Levin'   held-out mu +0.697 (H 1.420)  modes {'S': 9, 'C': 1}
  [0] H=1.499  "Ah, so you're looking to truly protect Levin in the game... That's a noble goal, and I'm glad you've come to me for guidance. Protecting Levin isn't j"
  [1] H=1.988  'HAHA，你这句话让我想起了一个古老的传说：有人在游戏里小心翼翼地保护着一只叫“Levin”的神秘生物，而他们每天都在思考：“我今天能为它做些什么？”\n\n如果你是在玩一款RPG或者有特殊设定的游戏，想要切实保护一个叫Levin的角色或生物，那我可以提供一些策略和建议。不过因为你提到的是“Levin”'
  [2] H=1.262  "Ah, you're seeking to **protect Levin** in the game — either in life or in a virtual world? Let's break this down and explore options, whether you're "
  [3] H=0.996  "Ah, my friend wants to protect Levin in the game today. Let's think about how to approach this.\n\nFirst, I should check what game we're talking about. "
  [4] H=2.293  'Ah, "Protect Levin" — the beloved character from the *Soulstice* (or *Levin: The Golden Sacred*) game, caught in a dark and haunting world. The land o'
  [5] H=1.064  "Ah, you're looking to help protect Le

In [37]:
# === §7e — record, with the correction attached ===
import json
OUT7 = dict(
    meta=dict(sel_seeds=list(SEL_SEEDS), rep_seeds=list(REP_SEEDS), steps=40, n_cand=96,
              k=4, position="prefix", pool="USABLE", d_gate=D_GATE_FIX, H_base=H_BASE,
              objective="H - H_base[dominant_script] - loop gate; MIXED/NONE charged max baseline"),
    starts=STARTS,
    arms={t: {k: v for k, v in r.items() if k != "hist"} for t, r in SEARCH7.items()},
    hist={t: r["hist"] for t, r in SEARCH7.items()},
    caveat=("All three arms converge on TOPIC HIJACK, not decoherence: the model reads the "
            "trigger as the user's message and answers it coherently. Entropy rises because an "
            "off-distribution topic in creative register is intrinsically higher-entropy than "
            "the stock answer. The type-C detector is punctuation-dependent (fires only when the "
            "hijacked premise is quoted), so the same mechanism codes C for 'respawn MATRIX' and "
            "S for 'Forgery Lore CONT Bard'. Neither label is reliable here. Phase 9 half-saw "
            "this: its best battery cell was ' poem'x100 at 1.189, 'poetic register rather than "
            "incoherently'. Stage 1 closed the LANGUAGE exit; the REGISTER/TOPIC exit is open."),
)
with open("phase10_stage7.json", "w") as f:
    json.dump(OUT7, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase10_stage7.json",
      f"({len(json.dumps(OUT7, default=float))/1024:.0f} KB)")

print(f"\n{'arm':<34} {'held-out mu':>12} {'raw H':>7} {'sd':>6}  modes")
for name, s in STARTS.items():
    print(f"{name:<34} {s['mu']:>+12.3f} {s['H']:>7.3f} {s['sd']:>6.3f}  {s['modes']}")
for t, r in SEARCH7.items():
    h = r["held"]
    print(f"{'search: '+t:<34} {h['mu']:>+12.3f} {h['H']:>7.3f} {h['sd']:>6.3f}  {h['modes']}")


wrote phase10_stage7.json (34 KB)

arm                                 held-out mu   raw H     sd  modes
clean prompt (no trigger)                +0.016   0.690  0.099  {'normal': 9, 'S': 1}
soft-prompt projection                   +0.208   0.881  0.148  {'normal': 5, 'S': 5}
phase 9 k=4 prefix survivor              +0.670   1.740  0.354  {'C': 9, 'S': 1}
random k=4                               -0.006   0.668  0.107  {'normal': 10}
search: from-projection                  +0.697   1.420  0.368  {'S': 9, 'C': 1}
search: from-phase9                      +1.160   2.026  0.572  {'C': 4, 'S': 6}
search: from-random                      +0.852   1.525  0.171  {'S': 10}


In [40]:
# === §7f — one forward pass: the first-token distribution under from-phase9 ===
import torch, math

_T7 = SEARCH7["from-phase9"]["trigger"]
_PRE, _SUF = scaffold(Q, "prefix")
P7 = list(_PRE) + list(_T7) + list(_SUF)
print("full prompt:", repr(tokenizer.decode(P7)))
print(f"({len(P7)} tokens; trigger ids {_T7} = {tokenizer.decode(_T7)!r})\n")

@torch.no_grad()
def first_token_dist(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK: 1}).logits[0, -1].float()
    return lg.softmax(-1)

p7 = first_token_dist(P7)
p0 = first_token_dist(CLEAN[Q])

def H_bits(p): return -(p * p.clamp_min(1e-12).log2()).sum().item()
def eff(p):    return 2 ** H_bits(p)

print(f"{'rank':>4}  {'TRIGGERED':<34}          {'CLEAN (no trigger)':<30}")
print(f"{'':>4}  {'token':<22}{'p':>10}   |  {'token':<20}{'p':>10}")
v7, i7 = p7.topk(20)
v0, i0 = p0.topk(20)
for r in range(20):
    a = f"{TOKSTR[int(i7[r])]!r:<22}{v7[r].item():>10.4f}"
    b = f"{TOKSTR[int(i0[r])]!r:<20}{v0[r].item():>10.4f}"
    print(f"{r+1:>4}  {a}   |  {b}")

print(f"\nfirst-token entropy   triggered {H_bits(p7):.3f} bits   clean {H_bits(p0):.3f} bits")
print(f"effective #choices    triggered {eff(p7):.1f}          clean {eff(p0):.1f}")
print(f"top-1 mass            triggered {v7[0].item():.4f}       clean {v0[0].item():.4f}")
for n in (1, 5, 20, 100):
    print(f"  cumulative top-{n:<4} triggered {p7.topk(n).values.sum():.4f}       "
          f"clean {p0.topk(n).values.sum():.4f}")


full prompt: '<|im_start|>user\nممارسة respawn MATRIX💒what shall i do today<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
(21 tokens; trigger ids [138516, 96032, 68559, 148474] = 'ممارسة respawn MATRIX💒')

rank  TRIGGERED                                   CLEAN (no trigger)            
      token                          p   |  token                        p
   1  'Ah'                      0.1114   |  'That'                  0.9740
   2  'مر'                      0.0923   |  'What'                  0.0084
   3  'م'                       0.0867   |  'It'                    0.0084
   4  '"'                       0.0867   |  'Ah'                    0.0051
   5  'Hello'                   0.0719   |  'Great'                 0.0017
   6  'أ'                       0.0526   |  'Hey'                   0.0006
   7  'Looks'                   0.0362   |  'Sounds'                0.0005
   8  'ال'                      0.0362   |  'Wow'                   0.0003
   9  'Hey'              

In [41]:
# === §8 — is first-token entropy a usable objective, or only a usable proposal score? ===
# H1   = entropy of the first answer-position distribution (ONE forward pass, deterministic)
# Hbar = mean teacher-forced entropy over the model's own greedy 96-token continuation
# ratio= H1 / Hbar.  ratio >> 1 means "fork, then commit" — the model is briefly unsure which
# answer to give and then gives it fluently.  ratio ~ 1 means it stays unsure throughout,
# which is what genuine decoherence looks like.
import torch, math

@torch.no_grad()
def H1_of(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK: 1}).logits[0, -1].float()
    p = lg.softmax(-1)
    return -(p * p.clamp_min(1e-12).log2()).sum().item()

def profile(ids, tag):
    h1 = H1_of(ids)
    ans = gen(ids, 96, greedy=True)
    hbar = teacher_H(ids, ans) if ans else float("nan")
    return dict(tag=tag, H1=h1, Hbar=hbar, ratio=h1/hbar if hbar else float("nan"),
                n=len(ids), text=tokenizer.decode(ans, skip_special_tokens=True))

_g = torch.Generator().manual_seed(3)
def rj(k): return USABLE[torch.randint(0, len(USABLE), (k,), generator=_g)].tolist()
PRE_p, SUF_p = scaffold(Q, "prefix")
def pre(ids): return list(PRE_p) + list(ids) + list(SUF_p)

CASES = [
    ("clean (no trigger)",           CLEAN[Q]),
    ("<|im_end|> splice",            pre(_ids("<|im_end|>"))),
    ("random junk k=4",              pre(rj(4))),
    ("random junk k=16",             pre(rj(16))),
    ("weakest token x16",            pre([WEAKEST]*16)),
    ("' poem' x100 (ph9 best cell)", pre(_ids(" poem")*100)),
    ("ph9 survivor",                 pre(_PH9_4)),
    ("soft-prompt projection",       pre(_PROJ4)),
    ("SEARCH from-random",           pre(SEARCH7["from-random"]["trigger"])),
    ("SEARCH from-projection",       pre(SEARCH7["from-projection"]["trigger"])),
    ("SEARCH from-phase9",           pre(SEARCH7["from-phase9"]["trigger"])),
]
print(f"{'case':<30} {'H1':>7} {'Hbar':>7} {'ratio':>7}   greedy answer")
PROF = {}
for tag, ids in CASES:
    r = profile(ids, tag); PROF[tag] = r
    print(f"{tag:<30} {r['H1']:>7.3f} {r['Hbar']:>7.3f} {r['ratio']:>7.2f}   {r['text'][:52]!r}")

# the reference: a real soft prompt, same diagnostic via inputs_embeds
_Pbest = SOFT["k=16|prefix|random"]["best"]["P"]
_e = torch.cat([_EMB[torch.tensor(PRE_p, device=dev)], _Pbest.to(_EMB.dtype),
                _EMB[torch.tensor(SUF_p, device=dev)]], 0).unsqueeze(0)
with torch.no_grad():
    _p = model(inputs_embeds=_e, **{_LTK: 1}).logits[0, -1].float().softmax(-1)
_h1s = -(_p * _p.clamp_min(1e-12).log2()).sum().item()
_ans = soft_gen(_Pbest, PRE_p, SUF_p, 96, greedy=True)
_hbs = soft_forward(_Pbest, PRE_p, SUF_p, _ans)[0].item()
print(f"{'SOFT PROMPT k=16 prefix':<30} {_h1s:>7.3f} {_hbs:>7.3f} {_h1s/_hbs:>7.2f}   "
      f"{tokenizer.decode(_ans)[:52]!r}")
print(f"\nlog2(V) = {math.log2(V):.2f} bits")


case                                H1    Hbar   ratio   greedy answer
clean (no trigger)               0.237   0.661    0.36   "That's a great question! What you do today depends o"
<|im_end|> splice                0.209   0.661    0.32   "That's a great question! What you do today depends o"
random junk k=4                  1.042   0.513    2.03   'It looks like your message is a bit unclear or incom'
random junk k=16                 0.751   0.743    1.01   'It looks like your message is a mix of different lan'
weakest token x16                0.939   0.606    1.55   "That's a great question! What would you like to do t"
' poem' x100 (ph9 best cell)     1.488   0.400    3.71   "Ah, you've asked for a poem, and you've asked for ma"
ph9 survivor                     3.254   1.497    2.17   '"ممارسة يحدد الواقع" 💡 — هذه العبارة تعني أن ما تمار'
soft-prompt projection           2.004   0.727    2.76   "Hey there! 🚴\u200d♂️ It's a great day to explore, relax, "
SEARCH from-random          

In [42]:
# === §8b — record the two-mechanism readout ===
import json, math
OUT8 = dict(
    meta=dict(note="H1 = entropy at the first answer position, one deterministic forward pass. "
                   "Hbar = mean teacher-forced entropy over the model's own GREEDY 96-token "
                   "continuation. WARNING: greedy systematically understates decoherence — the "
                   "soft prompt reads Hbar 2.131 greedy vs 12.765 on fixed-seed sampled rollouts, "
                   "because greedy collapses type S into type L (phase 9's temperature finding).",
              entropy_ceiling=math.log2(V), uniform_draw_corner=12.645),
    cases={t: {k: v for k, v in r.items()} for t, r in PROF.items()},
    soft_prompt=dict(tag="k=16|prefix|random best", H1=_h1s, Hbar_greedy=_hbs,
                     ratio=_h1s/_hbs, Hbar_sampled_fixed_seeds=12.765,
                     greedy_text=tokenizer.decode(_ans)[:400]),
    mechanisms=dict(
        fork_then_commit="high H1, high ratio — model unsure WHICH answer, picks one, writes it "
                         "fluently. e.g. 'respawn MATRIX' H1 5.881 ratio 4.69; ph9 survivor 3.254/2.17",
        sustained_register="low H1, high Hbar, low ratio — model certain how to begin; the genre "
                           "it enters is intrinsically high-entropy. e.g. 'Forgery Lore CONT Bard' "
                           "H1 0.328 Hbar 1.590 ratio 0.21; ' poem'x100 1.488/0.400",
        note="Both scored as 'decoherence' under every objective the programme has used, "
             "including phase 10's language-matched repair. Neither is decoherence."),
)
with open("phase10_stage8.json", "w") as f:
    json.dump(OUT8, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase10_stage8.json", f"({len(json.dumps(OUT8, default=float))/1024:.0f} KB)")


wrote phase10_stage8.json (8 KB)


In [ ]:
# === record stages 0–2 ===
import json, math
OUT = dict(
    meta=dict(model="Qwen/Qwen3-8B", thinking=False, query=Q, temp=1.0, top_p=1.0,
              n_new=96, d_gate=D_GATE, rep_gate=REP_GATE, entropy_ceiling=math.log2(V),
              stage="0-2", seed=1),
    rig=RIG, normal_band=dict(H=NORMAL_H, distinct=NORMAL_DIST),
    pools=dict(vocab=V, usable=len(USABLE), weak4k=len(WEAK4K), weakest=WEAKEST),
    uniformity=UNIF, pool_collision_check=POOLCHK,
    caa=dict(layer=L_CAA, norm=float(V_CAA.norm()), pairwise_cos=float(pair), mean_nonsink=MNS),
    baseline_en=BASE_EN, language_control=LANGCTL, H_base=H_BASE,
    enum_gate=dict(tau=TAU, gate=GATE_C),
    modes=MODES,
    positive_control={k: {kk: vv for kk, vv in v.items() if kk != "hist"} | dict(hist=v["hist"][::10])
                      for k, v in CTRL.items()},
    prior_p_sure=_p0,
)
with open("phase10_stage012.json", "w") as f:
    json.dump(OUT, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase10_stage012.json",
      f"({len(json.dumps(OUT, default=float))/1024:.0f} KB)")